# Minh Thang Trinh - Thesis - 2026

This notebook contains the code for the paper "Accelerating estimation for state-space models by weighted data subsampling" by Wang et al. (2025). 

The code is organised as follows:

- **Section 1** imports the relevant modules and defines functions. The function are divided into subsections, e.g. Section 1.3 defines model specific functions, while Section 1.4 defines functions specific to our methodology.
- **Section 2** contains a ARMA(1,1) simulated data example that illustrates the use of the functions in Section 1 and showcases the methodology.
- **Section 3** contains a empircal application in discretecized Vasicek model


## 1. Functions 
This section import modules and defines some functions used in the methodology.

### 1.1 Modules

In [ ]:
# Import modules
import matplotlib, time, copy
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.ticker as mticker
from contextlib import suppress
import matplotlib as mpl
from typing import Optional
import numpy as np
import random
from autograd import numpy as np_autograd
from autograd import jacobian as Jacobian_autograd
from autograd import grad, hessian
from autograd.scipy.special import gammaln as autograd_gammaln
from numdifftools import Gradient, Jacobian, Hessian
import scipy.stats as sps
from scipy.optimize import minimize, brentq, minimize_scalar, basinhopping
import tensorflow_probability.substrates.numpy as tfp
from scipy.special import gammaln, digamma, polygamma
import statsmodels.api as sm
import time
import re
from dataclasses import dataclass
from typing import Callable, Dict, List
from IPython.display import Markdown, display
from IPython.utils.capture import capture_output
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
import atexit
import inspect
import warnings
from __future__ import annotations
import json
import h5py
from collections.abc import Mapping
from statsmodels.tsa.arima_process import ArmaProcess   # WINSTON ADJUST
import autograd.scipy.stats as sps_autograd             # WINSTON ADJUST
import pandas_datareader.data as web # WINSTON ADJUST
import pandas as pd # WINSTON ADJUST

: 

In [ ]:
np.random.seed(1234)

### 1.2. Auxiliary functions

In [ ]:
def _robust_cholesky(S):
    """
    Robust Cholesky factorisation
    """
    try:
        return np.linalg.cholesky(S)
    except np.linalg.LinAlgError:
        jitter = 1e-12 * np.trace(S) / S.shape[0]
        return np.linalg.cholesky(S + jitter * np.eye(S.shape[0]))

def mvn_tail_sample(mean, cov, q=0.99, n=1, random_state=None):
    """
    To test the control variates for an extreme value of theta.
    
    Draw X ~ N(mean, cov) conditioned on being outside the q-quantile ellipse:
        (X-mean)^T cov^{-1} (X-mean) > chi2_{d, q}

    Uses SciPy's chi2 CDF/PPF to sample the truncated radius exactly (no rejection).
    """
    rng = np.random.default_rng(random_state)
    mean = np.asarray(mean)
    cov  = np.asarray(cov)
    d = mean.size

    # Cholesky for affine transform
    L = _robust_cholesky(cov)

    # Chi-square threshold and truncation point
    t_q = sps.chi2.ppf(q, df=d)
    F_t = sps.chi2.cdf(t_q, df=d)   # = q, but keep for clarity

    # Directions: uniform on the unit sphere (via normalise i.i.d. Gaussians)
    Z = rng.normal(size=(n, d))
    Udir = Z / np.linalg.norm(Z, axis=1, keepdims=True)

    # Radii: R^2 | R^2 > t_q  via inverse-CDF on the tail
    U = rng.random(n)
    R2 = sps.chi2.ppf(F_t + U * (1.0 - F_t), df=d)
    R = np.sqrt(R2)

    # Map to N(mean, cov)
    X = mean + (Udir * R[:, None]) @ L.T
    return X

def get_param_names(model_name, error_type=None, p=1, q=1):
    
    if model_name == "ARMA11":
        param_names = [r"$a_1$", r"$b_1$", r"$\sigma$"]
        trans_param_names = [r"$\widetilde{a}_1$", r"$\widetilde{b}_1$", r"$\widetilde{\sigma}$"]
    elif model_name == "Vasicek":
        param_names = [r"$a_1$", r"$\sigma$", r"$\tau$", r"$\mu$"]
        trans_param_names = [r"$\widetilde{a}_1$", r"$\widetilde{\sigma}$", r"$\widetilde{\tau}$", r"$\widetilde{\mu}$"]
    elif model_name == "Vasicek_Demeaned":
        param_names = [r"$a_1$", r"$\sigma$", r"$\tau$"]
        trans_param_names = [r"$\widetilde{a}_1$", r"$\widetilde{\sigma}$", r"$\widetilde{\tau}$"]
    else:
        raise ValueError
        
    if error_type == "Student-t":
        param_names = param_names + [r"$\nu$"]
        trans_param_names = trans_param_names + [r"$\widetilde{\nu}$"]
    
    return param_names, trans_param_names

def check_Hessian_close(H_analytical, H_numeric,
                       rtol=1e-6, atol=1e-8,
                       rel_tol_fro=1e-6, rel_tol_inf=1e-6,
                       max_report=20,
                       symmetrise=True,
                       print_if_pass=False):
    """
    Asked Chat-GPT to write a function to use for exploring differences in the two Hessians (analytical vs numerical)
    
    Diagnostic Hessian comparison: prints differences but NEVER raises.
    Returns (passed, report).

    Tolerance rule (per entry):
        |Ha - Hn| <= abs_atol + rtol*|Hn|,  with abs_atol = atol * max(1, ||Hn||_inf)

    Normwise checks:
        Frobenius and infinity-norm relative errors.

    Parameters
    ----------
    H_analytical, H_numeric : array_like
    rtol, atol : floats
    rel_tol_fro, rel_tol_inf : floats
    max_report : int
        Max violating entries to print (sorted by severity).
    symmetrise : bool
        If True, compare 0.5*(H + H.T) for both.
    print_if_pass : bool
        If False, prints nothing when all checks pass.

    Returns
    -------
    passed : bool
        True if entrywise + both normwise checks pass.
    report : dict
        Keys: fro_rel, inf_rel, num_bad, abs_atol, top (list of worst violations).
    """
    Ha = np.asarray(H_analytical, dtype=np.float64)
    Hn = np.asarray(H_numeric,     dtype=np.float64)

    if symmetrise:
        Ha = 0.5 * (Ha + Ha.T)
        Hn = 0.5 * (Hn + Hn.T)

    # Scale-aware absolute tolerance
    scale = max(1.0, np.linalg.norm(Hn, ord=np.inf))
    abs_atol = atol * scale

    # Normwise relative errors
    fro_rel = np.linalg.norm(Ha - Hn) / (np.linalg.norm(Hn) + 1e-16)
    inf_rel = (np.linalg.norm(Ha - Hn, ord=np.inf) /
               (np.linalg.norm(Hn, ord=np.inf) + 1e-16))

    # Entrywise violations
    tol_mat = abs_atol + rtol * np.abs(Hn)
    diff = Ha - Hn
    bad_mask = np.abs(diff) > tol_mat
    num_bad = int(np.count_nonzero(bad_mask))

    passed = (num_bad == 0) and (fro_rel <= rel_tol_fro) and (inf_rel <= rel_tol_inf)

    # Printing
    if (not passed) or print_if_pass:
        print("Hessian comparison (soft diagnostics)")
        print(f"- rtol={rtol:g}, atol={atol:g}, abs_atol={abs_atol:.3e}")
        print(f"- Frobenius relative error: {fro_rel:.3e} (thr {rel_tol_fro:.3e})")
        print(f"- Infinity-norm relative error: {inf_rel:.3e} (thr {rel_tol_inf:.3e})")
        print(f"- Entrywise violations: {num_bad}")

    top_rows = []
    if num_bad > 0:
        idxs = np.argwhere(bad_mask)
        strength = np.abs(diff[bad_mask]) / (tol_mat[bad_mask] + 1e-300)  # error / tol
        order = np.argsort(-strength)  # descending by severity
        nshow = min(max_report, num_bad)
        if not passed or print_if_pass:
            print(f"Top {nshow} violating entries (strength = |Δ| / tol):")
        for k in range(nshow):
            i, j = idxs[order[k]]
            ai = Ha[i, j]; ni = Hn[i, j]
            ae = abs(diff[i, j]); tj = tol_mat[i, j]; s = strength[order[k]]
            if not passed or print_if_pass:
                print(f"  (i={i}, j={j})  Ha={ai:.6e}, Hn={ni:.6e} | "
                      f"|Δ|={ae:.3e}, tol={tj:.3e}, strength={s:.2f}")
            top_rows.append({
                "i": int(i), "j": int(j),
                "Ha": float(ai), "Hn": float(ni),
                "abs_err": float(ae), "tol": float(tj),
                "strength": float(s)
            })
    elif print_if_pass:
        print("No entrywise violations under the given tolerances.")

    report = {
        "fro_rel": fro_rel,
        "inf_rel": inf_rel,
        "num_bad": num_bad,
        "abs_atol": abs_atol,
        "top": top_rows
    }
    return passed, report

"""
I asked ChatGPT to provide functions we need (such as h, hinv and its derivatives, and log-determinant of Jacobians)
for some standard transformations. It produced the below code (until the end of the notebook code block)

We create these as:

h, hinv, hinv_p, hinv_pp, jacdiag, logdet = initiate_transformation_functions(transforms)

where transforms contains a list with the different transforms we want to apply.
If no transforms are applied, then set all to 'identity'.
"""
# Autograd-friendly transforms and helpers
# Uses np_autograd everywhere (no plain numpy)

def _softplus(z):
    # stable two-branch form; both branches evaluated (OK for Autograd)
    return np_autograd.where(
        z > 0,
        z + np_autograd.log1p(np_autograd.exp(-z)),
        np_autograd.log1p(np_autograd.exp(z))
    )

def _logexpm1(y):
    # inverse of softplus
    return np_autograd.log(np_autograd.expm1(y))

def _sigmoid(z):
    return np_autograd.where(
        z >= 0,
        1.0 / (1.0 + np_autograd.exp(-z)),
        np_autograd.exp(z) / (1.0 + np_autograd.exp(z))
    )

def _logit(p):
    return np_autograd.log(p) - np_autograd.log1p(1.0 - p)

@dataclass(frozen=True)
class Transform:
    h: Callable[[object], object]            # θ -> ϕ
    hinv: Callable[[object], object]         # ϕ -> θ
    hinv_prime: Callable[[object], object]   # dθ/dϕ
    hinv_prime2: Callable[[object], object]  # d²θ/dϕ²
    theta_ok: Callable[[object], object]     # returns bool/array of bool
    phi_ok: Callable[[object], object]

REGISTRY: Dict[str, Transform] = {}

def register(name: str, t: Transform):
    if name in REGISTRY:
        raise ValueError(f"Transform '{name}' already registered.")
    REGISTRY[name] = t

def make_bounded(a: float, b: float) -> Transform:
    width = float(b - a)
    if not (np_autograd.isfinite(width) and width > 0):
        raise ValueError("Require b > a and both finite for bounded_(a,b).")

    def h(x):
        return np_autograd.log((x - a) / (b - x))

    def hinv(z):
        s = _sigmoid(z)
        return a + width * s

    def hinv_prime(z):
        s = _sigmoid(z)
        return width * s * (1.0 - s)

    def hinv_prime2(z):
        s = _sigmoid(z)
        return width * s * (1.0 - s) * (1.0 - 2.0 * s)

    return Transform(
        h=h, hinv=hinv, hinv_prime=hinv_prime, hinv_prime2=hinv_prime2,
        theta_ok=lambda x: (x > a) & (x < b),
        phi_ok=lambda z: np_autograd.ones_like(z, dtype=bool)
    )

def parse_and_maybe_register(name: str) -> str:
    if name in REGISTRY:
        return name
    m1 = re.fullmatch(r"bounded_(-?\d+(?:\.\d+)?)_(-?\d+(?:\.\d+)?)", name)
    m2 = re.fullmatch(r"bounded_\(\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\)", name)
    if m1 or m2:
        a = float((m1 or m2).group(1)); b = float((m1 or m2).group(2))
        register(name, make_bounded(a, b))
        return name
    l1 = re.fullmatch(r"logit_(-?\d+(?:\.\d+)?)_(-?\d+(?:\.\d+)?)", name)
    l2 = re.fullmatch(r"logit_\(\s*(-?\d+(?:\.\d+)?)\s*,\s*(-?\d+(?:\.\d+)?)\s*\)", name)
    if l1 or l2:
        a = float((l1 or l2).group(1)); b = float((l1 or l2).group(2))
        register(name, make_bounded(a, b))
        return name
    return name

# Built-ins (no casts/coercions)
register("identity", Transform(
    h=lambda x: x,
    hinv=lambda z: z,
    hinv_prime=lambda z: np_autograd.ones_like(z),
    hinv_prime2=lambda z: np_autograd.zeros_like(z),
    theta_ok=lambda x: np_autograd.ones_like(x, dtype=bool),
    phi_ok=lambda z: np_autograd.ones_like(z, dtype=bool),
))

register("log", Transform(
    h=lambda x: np_autograd.log(x),
    hinv=lambda z: np_autograd.exp(z),
    hinv_prime=lambda z: np_autograd.exp(z),
    hinv_prime2=lambda z: np_autograd.exp(z),
    theta_ok=lambda x: x > 0,
    phi_ok=lambda z: np_autograd.ones_like(z, dtype=bool),
))

register("log_shift2", Transform(
    h=lambda x: np_autograd.log(x - 2.0),
    hinv=lambda z: 2.0 + np_autograd.exp(z),
    hinv_prime=lambda z: np_autograd.exp(z),
    hinv_prime2=lambda z: np_autograd.exp(z),
    theta_ok=lambda x: x > 2.0,
    phi_ok=lambda z: np_autograd.ones_like(z, dtype=bool),
))

register("exp", Transform(
    h=lambda x: np_autograd.exp(x),
    hinv=lambda z: np_autograd.log(z),
    hinv_prime=lambda z: 1.0 / z,
    hinv_prime2=lambda z: -1.0 / (z * z),
    theta_ok=lambda x: np_autograd.ones_like(x, dtype=bool),
    phi_ok=lambda z: z > 0,
))

register("softplus", Transform(
    h=lambda x: _softplus(x),
    hinv=lambda z: _logexpm1(z),
    hinv_prime=lambda z: 1.0 / (1.0 - np_autograd.exp(-z)),
    hinv_prime2=lambda z: np_autograd.exp(-z) / (1.0 - np_autograd.exp(-z))**2,
    theta_ok=lambda x: np_autograd.ones_like(x, dtype=bool),
    phi_ok=lambda z: z > 0,
))

register("softplus_shift2", Transform(
    h=lambda x: _softplus(x - 2.0),
    hinv=lambda z: 2.0 + _logexpm1(z),
    hinv_prime=lambda z: 1.0 / (1.0 - np_autograd.exp(-z)),
    hinv_prime2=lambda z: np_autograd.exp(-z) / (1.0 - np_autograd.exp(-z))**2,
    theta_ok=lambda x: np_autograd.ones_like(x, dtype=bool),
    phi_ok=lambda z: z > 0,
))

register("logit", Transform(
    h=lambda x: _logit(x),
    hinv=lambda z: _sigmoid(z),
    hinv_prime=lambda z: (lambda s: s * (1.0 - s))(_sigmoid(z)),
    hinv_prime2=lambda z: (lambda s: s * (1.0 - s) * (1.0 - 2.0*s))(_sigmoid(z)),
    theta_ok=lambda x: (x > 0) & (x < 1),
    phi_ok=lambda z: np_autograd.ones_like(z, dtype=bool),
))

def initiate_transformation_functions(transform_names: List[str], validate: bool = False):
    """
    Returns callables: h, hinv, hinv_prime, hinv_prime2, jac_diag_hinv, log_abs_det_jac_hinv
    If validate=True, performs domain checks (avoid during AD).
    """
    keys = [parse_and_maybe_register(name) for name in transform_names]
    funcs = []
    for key in keys:
        if key not in REGISTRY:
            raise ValueError(f"Unknown transform '{key}'.")
        funcs.append(REGISTRY[key])
    p = len(funcs)

    def _apply_per_coord(vec, which: str):
        # List->stack avoids in-place writes
        return np_autograd.stack([getattr(funcs[i], which)(vec[i]) for i in range(p)], axis=0)

    def h(theta):
        if validate:
            for i, f in enumerate(funcs):
                ok = f.theta_ok(theta[i])
                if not bool(np_autograd.all(ok)):
                    raise ValueError(f"θ[{i}] violates domain for '{transform_names[i]}'")
        return _apply_per_coord(theta, "h")

    def hinv(phi):
        if validate:
            for i, f in enumerate(funcs):
                ok = f.phi_ok(phi[i])
                if not bool(np_autograd.all(ok)):
                    raise ValueError(f"ϕ[{i}] violates domain for inverse of '{transform_names[i]}'")
        return _apply_per_coord(phi, "hinv")

    def hinv_prime(phi):
        if validate:
            for i, f in enumerate(funcs):
                ok = f.phi_ok(phi[i])
                if not bool(np_autograd.all(ok)):
                    raise ValueError(f"ϕ[{i}] violates domain for inverse of '{transform_names[i]}'")
        return _apply_per_coord(phi, "hinv_prime")

    def hinv_prime2(phi):
        if validate:
            for i, f in enumerate(funcs):
                ok = f.phi_ok(phi[i])
                if not bool(np_autograd.all(ok)):
                    raise ValueError(f"ϕ[{i}] violates domain for inverse of '{transform_names[i]}'")
        return _apply_per_coord(phi, "hinv_prime2")

    def jac_diag_hinv(phi):
        return hinv_prime(phi)

    def log_abs_det_jac_hinv(phi):
        d = hinv_prime(phi)
        return np_autograd.sum(np_autograd.log(np_autograd.abs(d)))

    return h, hinv, hinv_prime, hinv_prime2, jac_diag_hinv, log_abs_det_jac_hinv

"""
Asked ChatGPT to write some code useful for saving figures and text as the notebook is executed.
"""

def pdf_add_text(text: str, title: Optional[str] = None,
                 pagesize=(8.27, 11.69), fontsize=10):
    """Add a text page to the open PDF."""
    fig = plt.figure(figsize=pagesize)
    y = 0.95
    if title:
        fig.text(0.06, y, title, fontsize=fontsize+2, weight="bold", va="top")
        y -= 0.05
    fig.text(0.06, y, text, family="monospace", fontsize=fontsize, va="top")
    PDF.savefig(fig, bbox_inches="tight")
    plt.close(fig)

def pdf_add_fig(fig=None, *, tight=True):
    """Add a figure (or all open figures) to the open PDF."""
    if fig is None:
        for num in plt.get_fignums():
            PDF.savefig(plt.figure(num), bbox_inches=("tight" if tight else None))
    else:
        PDF.savefig(fig, bbox_inches=("tight" if tight else None))

def pdf_close():
    """Close the PDF explicitly (optional; atexit will also close)."""
    PDF.close()
    print(f"Closed: {PDF_PATH}")

    
def print_kv(d):
    w = max(len(k) for k in d)
    for k, v in sorted(d.items()):
        if isinstance(v, float):
            print(f"{k:{w}} : {v:.6g}")
        else:
            print(f"{k:{w}} : {v}")

            
"""
Asked ChatGPT to do a pretty output of the object returned by Scipy's minimizer:

"""            

def print_minimize_summary(res, x_names=None, max_x=12, max_grad=12, latex=True, fmt=".3f"):
    """
    Nicely summarise a scipy.optimize.minimize result.
    - If latex=True (default) and in Jupyter, renders names/values with MathJax.
    - x_names can be LaTeX strings like r"$\\mu$", r"$\\omega$", etc.
    """
    # Try to enable LaTeX rendering in notebooks
    if latex:
        try:
            from IPython.display import display, Math, Markdown
        except Exception:
            latex = False  # no IPython display available

    # Helper: strip surrounding $...$ so names fit inside Math(...) cleanly
    def _strip_dollars(s):
        return s[1:-1] if isinstance(s, str) and len(s) >= 2 and s[0] == s[-1] == "$" else s

    # ---- Basics ----
    method = res.get('method', 'minimize') if hasattr(res, 'get') else 'minimize'
    print("=== Optimisation summary ===")
    print(f"Method     : {method}")
    print(f"Success    : {res.success}  (status {res.status})")
    print(f"Message    : {res.message}")
    print(f"Iterations : {res.get('nit', 'n/a')}   f-evals: {res.get('nfev', 'n/a')}   g-evals: {res.get('njev', 'n/a')}")
    print(f"Final f    : {res.fun:.10g}")

    # ---- Gradient info (uses res.jac) ----
    g = None
    if getattr(res, "jac", None) is not None:
        g = np.asarray(res.jac).ravel()
        gn2  = float(np.linalg.norm(g))
        gmax = float(np.max(np.abs(g)))
        if latex:
            display(Math(rf"\|\nabla f\|_2 = {gn2:.3e}\quad,\quad \max_i |\partial_i f| = {gmax:.3e}"))
        else:
            print(f"‖∇f‖₂      : {gn2:.3e}   max|∂f|: {gmax:.3e}")

    # ---- Solution vector ----
    x = np.asarray(res.x).ravel()
    n = x.size
    print(f"\nSolution x (n={n}):")

    # Build names (LaTeX-friendly by default)
    if x_names is None:
        names = [fr"$x_{i}$" for i in range(n)] if latex else [f"x{i}" for i in range(n)]
    else:
        names = list(x_names)

    # Show parameter names inline (LaTeX/Markdown) if not too many
    if latex and n <= 50:
        display(Markdown(", ".join(names)))
    else:
        # Plain text fallback
        head = min(n, max_x // 2)
        tail = max(0, n - max_x // 2)
        if n <= max_x:
            for nm, val in zip(names, x):
                print(f"  {nm}: {val:{fmt}}")
        else:
            for i in range(head):
                print(f"  {names[i]}: {x[i]:{fmt}}")
            print("  ...")
            for i in range(tail, n):
                print(f"  {names[i]}: {x[i]:{fmt}}")

    # Render aligned name–value pairs in LaTeX (first/last few if long)
    if latex:
        # Decide how many to show in the aligned block
        if n <= max_x:
            idxs = range(n)
        else:
            idxs = list(range(max_x // 2)) + ["..."] + list(range(n - max_x // 2, n))

        lines = []
        for idx in idxs:
            if idx == "...":
                lines.append(r"\vdots & \vdots")
                continue
            nm = _strip_dollars(names[idx])
            lines.append(fr"{nm} &= {x[idx]:{fmt}}")
        latex_block = r"\begin{aligned}" + r" \\ ".join(lines) + r"\end{aligned}"
        display(Math(latex_block))

    # ---- Gradient vector (optional) ----
    if g is not None:
        if latex:
            if n <= max_grad:
                lines_g = [fr"{_strip_dollars(names[i])} &= {g[i]:{fmt}}" for i in range(n)]
            else:
                half = max_grad // 2
                lines_g = [fr"{_strip_dollars(names[i])} &= {g[i]:{fmt}}" for i in range(half)]
                lines_g += [r"\vdots & \vdots"]
                lines_g += [fr"{_strip_dollars(names[i])} &= {g[i]:{fmt}}" for i in range(n - half, n)]
            display(Math(r"\text{Gradient at solution: }" + r"\;" + r"\begin{aligned}" + r" \\ ".join(lines_g) + r"\end{aligned}"))
        else:
            print("\nGradient at solution:")
            if n <= max_grad:
                for nm, gi in zip(names, g):
                    print(f"  {nm}: {gi:{fmt}}")
            else:
                half = max_grad // 2
                for i in range(half):
                    print(f"  {names[i]}: {g[i]:{fmt}}")
                print("  ...")
                for i in range(n - half, n):
                    print(f"  {names[i]}: {g[i]:{fmt}}")

    # ---- Inverse-Hessian info ----
    if hasattr(res, "hess_inv") and res.hess_inv is not None:
        print("\nHessian^-1: L-BFGS product object available (use res.hess_inv.dot(v)).")
        
"""
Asked ChatGPT to do a callback for optim we can call to keep track of how the optimisation proceeds.
"""        
class FGCache:
    def __init__(self, f, g):
        self.f_raw, self.g_raw = f, g
        self.f = None; self.g = None
    def fun(self, x):
        self.f = self.f_raw(x); return self.f
    def jac(self, x):
        self.g = self.g_raw(x); return self.g

def callback_optim(xk, *cb_args):
    """
    Supports:
      - SciPy <=1.10: callback(xk, f, g)
      - SciPy >=1.11: callback(xk) or callback(xk, state)
    """
    
    params = ", ".join(f"{v:.2f}" for v in np.ravel(xk))
    params_theta = ", ".join(f"{v:.2f}" for v in np.ravel(hinv(xk)))
    
    it["k"] += 1
    f = g = None
    if len(cb_args) >= 2:                      # (xk, f, g)
        f, g = cb_args[0], cb_args[1]
    elif len(cb_args) == 1 and hasattr(cb_args[0], "fun"):  # (xk, state)
        f, g = cb_args[0].fun, cb_args[0].grad
    else:                                      # (xk) only → use cache
        f, g = fg.f, fg.g
    gn = np.linalg.norm(g) if g is not None else float("nan")
    fv = f if f is not None else float("nan")
    print(f"it {it['k']:4d}  f={fv:.6e}  ||g||={gn:.3e} \n phi=[{params}] \n theta = [{params_theta}]")

    
"""
Functions to plot/compute output from MCMC/Subsampling MCMC VB/Subsampling VB
"""
def plot_credible_intervals(param_draws, param_true=None, param_names=None, use_median=False):
    """
    param_draws: (M, p) array of posterior draws
    param_true:  (p,) array of true values (optional)
    param_names: list of length p for x tick labels (optional)
    use_median: show posterior medians (True) or means (False)
    """
    param_draws = np.asarray(param_draws)
    if param_true is not None:
        param_true  = np.asarray(param_true)

    # 95% intervals (your title says 95%)
    q_lo = np.quantile(param_draws, 0.025, axis=0)
    q_hi = np.quantile(param_draws, 0.975, axis=0)
    centre = (np.quantile(param_draws, 0.5, axis=0) if use_median
              else param_draws.mean(axis=0))

    p = param_draws.shape[1]
    xs = np.arange(p)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.vlines(xs, q_lo, q_hi, linewidth=2, label="95% credible interval")
    ax.plot(xs, centre, "o", label=("median" if use_median else "mean"))

    # plot true values if provided
    if param_true is not None:
        ax.plot(xs, param_true, "x", markersize=8, label="true")

    ax.set_xlabel("parameter" if (param_names is not None and len(param_names)==p) else "parameter index")
    ax.set_ylabel("value")
    ax.set_title("Posterior 95% credible intervals vs true values")
    ax.xaxis.set_major_locator(mticker.MultipleLocator(1))

    if param_names is not None and len(param_names) == p:
        # Option A (backwards-compatible):
        ax.set_xticks(xs)
        ax.set_xticklabels(param_names)  # <-- this avoids the TypeError
        # Option B (newer mpl only): ax.set_xticks(xs, labels=param_names)
    else:
        ax.set_xticks(xs)

    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5),
              borderaxespad=0., frameon=True)
    fig.tight_layout()
    return fig, ax

def plot_traceplots_MCMC(param_draws, 
                    param_true=None, 
                    param_names=None, 
                    burnin=0, 
                    thin=1, 
                    use_median=False, 
                    show_running_mean=True, 
                    runmean_window=50, 
                    suptitle="Traceplots"):
    """
    Plot traceplots of the MCMC chain for each parameter, with an optional line for the true value
    and a horizontal line for the posterior centre (median or mean).

    Parameters
    ----------
    param_draws : array_like, shape (M, p)
        Posterior draws (rows = iterations, columns = parameters).
    param_true : array_like, shape (p,), optional
        True values to mark as horizontal lines (one per parameter).
    param_names : list of str, optional
        Names for parameters (length p). If None, uses indices.
    burnin : int, default 0
        Number of initial iterations to discard.
    thin : int, default 1
        Thinning factor (keep every 'thin' draw after burn-in).
    use_median : bool, default True
        If True, draw horizontal line at posterior median; else posterior mean.
    show_running_mean : bool, default True
        Overlay a running mean to assess mixing visually.
    runmean_window : int, default 50
        Window size for the running mean (in kept iterations).
    suptitle : str, default "Traceplots"
        Figure suptitle.

    Returns
    -------
    fig, axes : matplotlib Figure and Axes array
    """
    D = np.asarray(param_draws)
    if D.ndim != 2:
        raise ValueError("param_draws must be a 2D array of shape (M, p).")
    M, p = D.shape

    # Slice for burn-in and thinning
    Dk = D[burnin::thin, :]
    Mk = Dk.shape[0]

    # Names and truths
    if param_names is None or len(param_names) != p:
        param_names = [f"θ{j+1}" for j in range(p)]
    if param_true is not None:
        param_true = np.asarray(param_true)
        if param_true.shape != (p,):
            raise ValueError("param_true must have shape (p,) if provided.")

    # Posterior centres
    centres = np.median(Dk, axis=0) if use_median else Dk.mean(axis=0)

    # Layout: one row per parameter
    h_per_ax = 1.6
    fig, axes = plt.subplots(
        nrows=p, ncols=1, figsize=(10, max(2.5, h_per_ax*p)), sharex=True
    )
    if p == 1:
        axes = np.array([axes])

    iters = np.arange(Mk)  # iteration index after burn-in/thin

    for j, ax in enumerate(axes):
        y = Dk[:, j]

        # Main trace
        ax.plot(iters, y, lw=0.8, alpha=0.9)
        ax.set_ylabel(param_names[j])

        # True value (if provided)
        if param_true is not None:
            ax.axhline(param_true[j], linestyle="--", linewidth=1.5, color = "lightcoral", label="true")

        # Posterior centre
        ax.axhline(centres[j], linestyle="-.", linewidth=1.2, label=("median" if use_median else "mean"))

        # Running mean
        if show_running_mean and Mk >= max(5, runmean_window):
            # Simple cumulative mean is often more informative for convergence
            cmean = np.cumsum(y) / (np.arange(Mk) + 1)
            ax.plot(iters, cmean, linewidth=1.1, alpha=0.9, label="running mean")

        # Neat ticks
        ax.yaxis.set_major_locator(mticker.MaxNLocator(4))

        # Minimal legend – only on first axis to avoid clutter
        if j == 0:
            ax.legend(loc="upper right", frameon=True)

    axes[-1].set_xlabel("iteration (including burn-in)")
    fig.suptitle(suptitle)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    return fig, axes

def plot_subsampling_MCMC_quantities(result_subsampling_MCMC, result_MCMC = None):
    """
    Plots some subsampling MCMC quantities. If result_MCMC is supplied, then the kernel density
    estimates for both methods are plotted in the same graph.
    """
    plt.figure(figsize=(8, 3.5))
    plt.title(f"Estimated log-posterior over iterations")
    iterations = range(result_subsampling_MCMC['N'] + 1) # starting value also stored 
    plt.plot(iterations, result_subsampling_MCMC['log_posthat_samples'], color="cornflowerblue")
    plt.xlabel(r"MCMC iteration")
    plt.ylabel(r"$\widehat{\ell}(\theta) + \log \pi(\theta)$")
    plt.tight_layout()

    plt.figure(figsize=(8, 3.5))
    plt.title(f"Estimated variance of log-likelihood estimator over iterations")
    iterations = range(result_subsampling_MCMC['N'] + 1) # starting value also stored 
    plt.plot(iterations, result_subsampling_MCMC['sigma2_lhat_samples'], color="cornflowerblue")
    plt.xlabel(r"MCMC iteration")
    plt.ylabel(r"$\widehat{\sigma}^(\theta)$")
    plt.yscale("log")
    plt.tight_layout()
    
    plt.figure(figsize=(8, 3.5))
    plt.title(r"$u_{\max}$ at proposed samples over iterations")
    plt.yscale("log")
    iterations = range(1, result_subsampling_MCMC['N'] + 1) 
    plt.plot(iterations, result_subsampling_MCMC['u_max_prop'], color="cornflowerblue", alpha = 0.5)
    plt.xlabel(r"MCMC iteration")
    plt.ylabel(r"$u_{\max}$ of $\mathbf{u}_p$")
    plt.xlim(0, 1.3*result_subsampling_MCMC['N'])
    plt.hlines(np.mean(result_subsampling_MCMC['u_max_prop']), xmin = 0, xmax = result_subsampling_MCMC['N'] + 1, linestyle="--", linewidth=1.5, color = "lightcoral", label="Average")
    plt.hlines(np.median(result_subsampling_MCMC['u_max_prop']), xmin = 0, xmax = result_subsampling_MCMC['N'] + 1, linestyle="-", linewidth=1.5, color = "forestgreen", label="Median")
    plt.legend()
    plt.tight_layout()
    
    
def plot_inefficiency_comparison(tau_full, tau_sub, param_names=None, 
                                 title="Inefficiency factors: Full vs Subsampling"):
    """
    Side-by-side barplot comparing inefficiency factors for each parameter.
    
    tau_full : array-like, shape (d,)
        Inefficiency factors from full-data MCMC.
    tau_sub : array-like, shape (d,)
        Inefficiency factors from subsampling MCMC.
    param_names : list of strings, optional
        Names of the parameters. If None, parameters are indexed.
    """
    
    tau_full = np.asarray(tau_full)
    tau_sub  = np.asarray(tau_sub)
    d = len(tau_full)

    if param_names is None:
        param_names = [f"θ{j+1}" for j in range(d)]

    x = np.arange(d)
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))

    bars1 = ax.bar(x - width/2, tau_full, width, label="Full data", alpha=0.8)
    bars2 = ax.bar(x + width/2, tau_sub,  width, label="Subsampling", alpha=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(param_names)
    ax.set_ylabel("Inefficiency factor (τ)")
    ax.set_title(title)
    ax.legend()

    # Optionally add bar labels
    for bars in [bars1, bars2]:
        for b in bars:
            h = b.get_height()
            ax.annotate(f"{h:.1f}",
                        xy=(b.get_x() + b.get_width()/2, h),
                        xytext=(0, 3),
                        textcoords="offset points",
                        ha="center", va="bottom", fontsize=8)

    plt.tight_layout()

def plot_kde_comparison(
    samples_1,
    samples_2,
    param_names=None,
    title="Posterior marginals: density 1 vs density 2",
    n_points=200,
    true_vals=None,
    labels=("Density 1", "Density 2"),
    bw_method="scott",
    bw_adjust=1.0,
):
    """
    Compare KDEs of two sets of samples (e.g. MCMC vs VI) for each parameter.

    Parameters
    ----------
    samples_1 : array-like, shape (n1, d) or (n1,)
        Draws from the first distribution.
    samples_2 : array-like, shape (n2, d) or (n2,)
        Draws from the second distribution.
    param_names : list of str, optional
        Names of parameters. If None, parameters are indexed as θ1, θ2, ...
    title : str
        Overall title for the figure.
    n_points : int
        Number of grid points for KDE evaluation.
    true_vals : float or array-like, optional
        True parameter values. If scalar, the same value is used for all
        parameters. If array-like, must have length d.
    labels : tuple of str, default ("Density 1", "Density 2")
        Legend labels for the two KDEs.
    bw_method : str, scalar or callable, default "scott"
        Passed to scipy.stats.gaussian_kde as the bandwidth method
        (e.g. "scott", "silverman", or a scalar).
    bw_adjust : float, default 1.0
        Multiplicative adjustment to the KDE bandwidth.
        >1.0 → smoother; <1.0 → more wiggly.

    Returns
    -------
    fig, axes : matplotlib Figure and Axes array
    """
    samples_1 = np.asarray(samples_1, dtype=float)
    samples_2 = np.asarray(samples_2, dtype=float)

    # Ensure 2D
    if samples_1.ndim == 1:
        samples_1 = samples_1[:, None]
    if samples_2.ndim == 1:
        samples_2 = samples_2[:, None]

    n1, d = samples_1.shape
    n2, d2 = samples_2.shape
    if d2 != d:
        raise ValueError("samples_1 and samples_2 must have same number of parameters (columns).")

    if param_names is None:
        param_names = [f"θ{j+1}" for j in range(d)]

    # Handle true values
    if true_vals is not None:
        true_vals = np.asarray(true_vals, dtype=float)
        if true_vals.ndim == 0:
            true_vals = np.repeat(true_vals, d)
        elif true_vals.shape[0] != d:
            raise ValueError("true_vals must be scalar or have length equal to the number of parameters d.")

    # Layout of subplots
    n_cols = min(d, 3)
    n_rows = int(np.ceil(d / n_cols))

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4 * n_cols, 3 * n_rows),
        squeeze=False
    )
    axes = axes.ravel()

    label1, label2 = labels

    for j in range(d):
        ax = axes[j]

        x1 = samples_1[:, j]
        x2 = samples_2[:, j]
        x_all = np.concatenate([x1, x2])

        # KDEs
        kde1 = sps.gaussian_kde(x1, bw_method=bw_method)
        kde2 = sps.gaussian_kde(x2, bw_method=bw_method)

        # Optional bandwidth adjustment
        if bw_adjust != 1.0:
            kde1.set_bandwidth(bw_method=kde1.factor * bw_adjust)
            kde2.set_bandwidth(bw_method=kde2.factor * bw_adjust)

        # Common grid with small padding
        x_min, x_max = np.min(x_all), np.max(x_all)
        pad = 0.05 * (x_max - x_min if x_max > x_min else 1.0)
        x_grid = np.linspace(x_min - pad, x_max + pad, n_points)

        y1 = kde1(x_grid)
        y2 = kde2(x_grid)

        ax.plot(x_grid, y1, label=label1, alpha=0.9)
        ax.plot(x_grid, y2, label=label2, alpha=0.9, linestyle="--")

        # True value line, if provided
        if true_vals is not None:
            ax.axvline(true_vals[j], color="k", linestyle=":", linewidth=1.2)

        ax.set_title(param_names[j])
        ax.set_ylabel("Density")
        ax.set_xlabel("Parameter value")

    # Remove any unused axes
    for k in range(d, len(axes)):
        fig.delaxes(axes[k])

    # Global legend
    handles, legend_labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, legend_labels, loc="upper right")
    fig.suptitle(title, y=0.98)
    plt.tight_layout(rect=[0, 0, 0.92, 0.95])

    return fig, axes

def _autocovariance_fft(x):
    """
    Fast autocovariance estimator using FFT.

    Parameters
    ----------
    x : array_like, shape (n,)
        1D time series / MCMC chain.

    Returns
    -------
    acov : ndarray, shape (n,)
        Autocovariance sequence starting at lag 0.
    """
    x = np.asarray(x, dtype=float)
    n = x.size
    x = x - x.mean()

    # Zero-pad to next power of 2 >= 2n for speed and to reduce circular artefacts
    n_fft = 1 << (2 * n - 1).bit_length()

    fx = np.fft.rfft(x, n=n_fft)
    acov = np.fft.irfft(fx * np.conjugate(fx), n=n_fft)[:n]
    acov /= n  # overall scale cancels when we normalise, so this is fine
    return acov


def IACT(x, return_acov=False):
    """
    Estimate integrated autocorrelation time (inefficiency factor)
    using Geyer's initial positive sequence and an FFT autocovariance.

    Parameters
    ----------
    x : array_like, shape (n,)
        1D MCMC chain.
    return_acov : bool, default False
        If True, also return the autocovariance and autocorrelation arrays.

    Returns
    -------
    tau_int : float
        Estimated integrated autocorrelation time (inefficiency factor).

    acov : ndarray, optional
        Autocovariance sequence (only if return_acov=True).

    acorr : ndarray, optional
        Autocorrelation sequence (only if return_acov=True).
    """
    x = np.asarray(x, dtype=float)
    n = x.size
    if n < 2:
        raise ValueError("Need at least two samples")

    acov = _autocovariance_fft(x)
    acorr = acov / acov[0]

    # Geyer: work with Gamma_k = rho_{2k} + rho_{2k+1}
    # and keep adding while Gamma_k > 0
    tau = 1.0
    for k in range(1, n - 1, 2):
        gamma = acorr[k] + acorr[k + 1]
        if gamma <= 0:
            break
        tau += 2.0 * gamma

    if return_acov:
        return tau, acov, acorr
    return tau


"""
ChatGPT functions to save results
"""
def _jsonify(x):
    """Make objects JSON-serialisable, including nested NumPy + mappings."""
    if isinstance(x, np.generic):
        return x.item()               # NumPy scalar -> Python scalar
    if isinstance(x, np.ndarray):
        return x.tolist()             # NumPy array -> list
    if isinstance(x, Mapping):
        return {str(k): _jsonify(v) for k, v in x.items()}  # dict-like (incl. OptimizeResult)
    if isinstance(x, (list, tuple)):
        return [_jsonify(v) for v in x]
    return x

def save_dict_h5(path, data, compression="gzip", level=4):
    """
    Minimal saver: arrays -> /arrays/<key>; everything else -> /meta_json.
    """
    path = Path(path).with_suffix(".h5")
    meta = {}
    arrays = {}
    for k, v in data.items():
        if isinstance(v, np.ndarray):
            arrays[k] = v
        else:
            meta[k] = _jsonify(v)

    with h5py.File(path, "w") as f:
        if arrays:
            g = f.create_group("arrays")
            for k, arr in arrays.items():
                g.create_dataset(k, data=arr, compression=compression, compression_opts=level)
        f.create_dataset("meta_json", data=np.void(json.dumps(meta).encode("utf-8")))
    return path

def load_dict_h5(path):
    """Minimal loader to round-trip the structure above."""
    path = Path(path)
    out = {}
    with h5py.File(path, "r") as f:
        # restore meta
        meta = json.loads(bytes(f["meta_json"][()]).decode("utf-8"))
        out.update(meta)
        # restore arrays (if any)
        if "arrays" in f:
            for k in f["arrays"].keys():
                out[k] = f["arrays"][k][()]
    return out

def pack_optimize_result(res):
    out = {}
    for k, v in dict(res).items():  # OptimizeResult is dict-like
        if k in {"hess_inv"}:  # often not JSON-safe / huge
            continue
        if isinstance(v, np.ndarray):
            out[k] = v.tolist()
        elif isinstance(v, (str, bool, int, float)) or v is None:
            out[k] = v
        elif isinstance(v, np.generic):
            out[k] = v.item()
        else:
            out[k] = str(v)  # last resort
    return out

### 1.3 Model functions
This section contains functions specific to the model, such as log-density, log-posterior, etc.

In [ ]:
def simulate_ARMA11(T, theta):
    # Define AR and MA coefficients
    ar = np_autograd.array([1, -theta[0]])  
    ma = np_autograd.array([1, -theta[1]])         

    # Create ARMA process object
    arma_process = ArmaProcess(ar, ma)

    # Simulate T samples
    y = arma_process.generate_sample(nsample=T, scale=theta[2]) # scale is the variance of the white noise

    return y

def initiate_KF_parameter_values_ARMA11(theta):
    """
    Initialise the parameters for the Kalman Filter based on Winston's analytical gradient and Hessian for ARMA(1,1)
    The general version for ARMA(p,q) could be found in the paper "Computation of the Gradient and the Hessian of the Log-likelihood of the
    State-space Model by the Kalman Filter"

    Input:
    theta: vector of parameters
        - a: AR coefficient
        - b: MA coefficient
        - sgm: standard deviation of the white noise

    Output is the gradient and Hessian of matricies F, G, H, Q and covariance matrix V
    """

    # Initial values for Kalman Filter
    a, b, sgm = theta
    d = len(theta)
    k = 2
    m = 1

    # Simple operations that autograd can handle
    F = np_autograd.array([[a, 1.0], [0.0, 0.0]])
    G = np_autograd.array([[1.0], [-b]]).reshape(2,1)
    H = np_autograd.array([[1.0, 0.0]]).reshape(1,2)
    Q = np_autograd.array([[sgm**2]])
    R = 0

    """
    1. Gradient parts
    """
    
    # Declare the gradient of F
    dF = np_autograd.zeros((d, 1, k, k))
    dF[0, 0, 0, 0] = 1.0

    # Declare the gradient of G
    dG = np_autograd.zeros((d, 1, k, 1))
    dG[1, 0, 1, 0] = -1.0

    dH = np_autograd.zeros((d, 1, 1, k))

    # Declare the gradient of Q
    dQ = np_autograd.zeros((d, 1, 1, 1))
    dQ[2, 0, 0, 0] = 2 * sgm

    # Declare the gradient of R
    dR = np_autograd.zeros((d, 1, 1, 1))

    # Compute the initial value of covariance matrix V
    g = np_autograd.array([1.0, a - b, a * (a - b)])

    C = np_autograd.array([
        sgm**2 * (1 - 2 * a * b + b**2) / (1 - a**2),
        a * (sgm**2 * (1 - 2 * a * b + b**2) / (1 - a**2)) - sgm**2 * b,
        a * (a * (sgm**2 * (1 - 2 * a * b + b**2) / (1 - a**2)) - sgm**2 * b)
    ])

    dV = np_autograd.zeros((d, 1, k, k))
    V = np_autograd.array([
        [C[0], -b * g[0]],
        [-b * g[0], b**2 * sgm**2]
    ])

    dVa = np_autograd.array([
        [(2 * sgm**2 * (a-b) * (1 - a*b)) / (1 - a**2)**2, 0],
        [0, 0]
    ])
    dVb = np_autograd.array([
        [2 * sgm**2 * (b-a) / (1 - a**2), -1],
        [-1, 2 * b * sgm**2]
    ])
    dVs = np_autograd.array([
        [2 * sgm * (1 - 2 * a * b + b**2) / (1 - a**2), 0],
        [0, 2 * b**2 * sgm]
    ])

    dV[0, 0, :, :] = dVa
    dV[1, 0, :, :] = dVb
    dV[2, 0, :, :] = dVs


    """
    2. Hessian parts
    """

    # Declare the Hessian of F, G, H, Q
    d2F = np_autograd.zeros((d, d, k, k))
    d2G = np_autograd.zeros((d, d, k, 1))
    d2H = np_autograd.zeros((d, d, 1, k))
    d2Q = np_autograd.zeros((d, d, 1, 1))
    d2Q[2, 2, 0, 0] = 2.0
    d2R = np_autograd.zeros((d, d, 1, 1))

    # Declare the Hessian of covariance matrix V
    d2V = np_autograd.zeros((d, d, k, k))

    d2V[0, 0, 0, 0] = -2*(-1 + 6*a*b + 2*a**3 *b - b**2 - 3* a**2 *(1+b**2)) * (sgm**2) / ((1-a**2)**3)
    d2V[0, 1, 0, 0] = -2 * (1 + a**2 - 2*a*b) * (sgm**2) / ((1-a**2)**2)
    d2V[0, 2, 0, 0] = -4 * (b + a**2 * b - a*(1+b**2)) * sgm / (1-a**2)**2

    d2V[1, 1, 0, 0] = 2*sgm**2 / (1-a**2)
    d2V[1, 1, 1, 1] = 2*sgm**2

    d2V[1, 2, 0, 0] = 2*sgm*(-2 * a + 2 * b) / (1-a**2)
    d2V[1, 2, 1, 1] = 4*b*sgm
    
    d2V[2, 2, 0, 0] = 2*(1-2*a*b + b**2) / (1-a**2)
    d2V[2, 2, 1, 1] = 2 * b**2
    
    # Copy the same Hessian
    d2V[1, 0, :, :] = d2V[0, 1, :, :].copy()
    d2V[2, 0, :, :] = d2V[0, 2, :, :].copy()
    d2V[2, 1, :, :] = d2V[1, 2, :, :].copy()

    # Initialize the x and dx
    x = np_autograd.zeros((k, 1))
    dx = np_autograd.zeros((d, 1, k, 1))
    d2x = np_autograd.zeros((d, d, k, 1))

    return F, G, H, Q, R, dF, dG, dH, dQ, dR, d2F, d2G, d2H, d2Q, d2R, V, dV, d2V, x, dx, d2x, k, d, m


def initiate_KF_parameter_values_Vasicek(theta):
    """
    Initialise the parameters for the Kalman Filter based on Winston's analytical gradient and Hessian for AR(1) with mean and 2 noise terms
    The general version for ARMA(p,q) could be found in the paper "Computation of the Gradient and the Hessian of the Log-likelihood of the
    State-space Model by the Kalman Filter"

    Input:
    theta: vector of parameters
        - a: AR coefficient
        - b: MA coefficient
        - sgm: standard deviation of the white noise

    Output is the gradient and Hessian of matricies F, G, H, Q and covariance matrix V
    """

    # Initial values for Kalman Filter
    a, sgm, tau, c = theta
    d = len(theta)
    k = 1
    m = 1

    # Simple operations that autograd can handle
    F = np_autograd.array([[a]])
    G = np_autograd.array([[1.0]]).reshape(1,1)
    H = np_autograd.array([[1.0]]).reshape(1,1)
    Q = np_autograd.array([[sgm ** 2]])
    R = tau ** 2

    """
    1. Gradient parts
    """
    
    # Declare the gradient of F
    dF = np_autograd.zeros((d, 1, k, k))
    dF[0, 0, 0, 0] = 1.0

    # Declare the gradient of G
    dG = np_autograd.zeros((d, 1, k, 1))
    dH = np_autograd.zeros((d, 1, 1, k))

    # Declare the gradient of Q
    dQ = np_autograd.zeros((d, 1, 1, 1))
    dQ[1, 0, 0, 0] = 2 * sgm

    # Declare the gradient of R
    dR = np_autograd.zeros((d, 1, 1, 1))
    dR[2, 0, 0, 0] = 2 * tau

    # Compute the initial value of covariance matrix V
    V = np_autograd.array([[sgm**2 / (1 - a**2)]])
    dV = np_autograd.zeros((d, 1, 1, 1))
    dV[0, 0, 0, 0] = (2 * a * sgm**2) / ((1 - a**2)**2)
    dV[1, 0, 0, 0] = 2 * sgm / (1 - a**2)


    """
    2. Hessian parts
    """

    # Declare the Hessian of F, G, H, Q
    d2F = np_autograd.zeros((d, d, k, k))
    d2G = np_autograd.zeros((d, d, k, 1))
    d2H = np_autograd.zeros((d, d, 1, k))
    d2Q = np_autograd.zeros((d, d, 1, 1))
    d2R = np_autograd.zeros((d, d, 1, 1))
    d2Q[1, 1, 0, 0] = 2.0
    d2R[2, 2, 0, 0] = 2.0

    # Declare the Hessian of covariance matrix V
    d2V = np_autograd.zeros((d, d, k, k))
    d2V[0, 0, 0, 0] = 2 * sgm**2 * (1 + 3 * a**2) / (1 - a**2)**3
    d2V[1, 1, 0, 0] = 2 / (1 - a**2)
    d2V[0, 1, 0, 0] = 4 * a * sgm / (1 - a**2)**2
    d2V[1, 0, 0, 0] = 4 * a * sgm / (1 - a**2)**2

    # Initialize the x and dx
    x = np_autograd.zeros((k, 1))
    dx = np_autograd.zeros((d, 1, k, 1))
    d2x = np_autograd.zeros((d, d, k, 1))

    return F, G, H, Q, R, dF, dG, dH, dQ, dR, d2F, d2G, d2H, d2Q, d2R, V, dV, d2V, x, dx, d2x, k, d, m


def initiate_KF_parameter_values_Vasicek_Demeaned(theta):
    """
    Initialise the parameters for the Kalman Filter based on Winston's analytical gradient and Hessian for AR(1) with mean and 2 noise terms
    The general version for ARMA(p,q) could be found in the paper "Computation of the Gradient and the Hessian of the Log-likelihood of the
    State-space Model by the Kalman Filter"

    Input:
    theta: vector of parameters
        - a: AR coefficient
        - b: MA coefficient
        - sgm: standard deviation of the white noise

    Output is the gradient and Hessian of matricies F, G, H, Q and covariance matrix V
    """

    # Initial values for Kalman Filter
    a, sgm, tau = theta
    d = len(theta)
    k = 1
    m = 1

    # Simple operations that autograd can handle
    F = np_autograd.array([[a]])
    G = np_autograd.array([[1.0]]).reshape(1,1)
    H = np_autograd.array([[1.0]]).reshape(1,1)
    Q = np_autograd.array([[sgm ** 2]])
    R = tau ** 2

    """
    1. Gradient parts
    """
    
    # Declare the gradient of F
    dF = np_autograd.zeros((d, 1, k, k))
    dF[0, 0, 0, 0] = 1.0

    # Declare the gradient of G
    dG = np_autograd.zeros((d, 1, k, 1))
    dH = np_autograd.zeros((d, 1, 1, k))

    # Declare the gradient of Q
    dQ = np_autograd.zeros((d, 1, 1, 1))
    dQ[1, 0, 0, 0] = 2 * sgm

    # Declare the gradient of R
    dR = np_autograd.zeros((d, 1, 1, 1))
    dR[2, 0, 0, 0] = 2 * tau

    # Compute the initial value of covariance matrix V
    V = np_autograd.array([[sgm**2 / (1 - a**2)]])
    dV = np_autograd.zeros((d, 1, 1, 1))
    dV[0, 0, 0, 0] = (2 * a * sgm**2) / ((1 - a**2)**2)
    dV[1, 0, 0, 0] = 2 * sgm / (1 - a**2)


    """
    2. Hessian parts
    """

    # Declare the Hessian of F, G, H, Q
    d2F = np_autograd.zeros((d, d, k, k))
    d2G = np_autograd.zeros((d, d, k, 1))
    d2H = np_autograd.zeros((d, d, 1, k))
    d2Q = np_autograd.zeros((d, d, 1, 1))
    d2R = np_autograd.zeros((d, d, 1, 1))
    d2Q[1, 1, 0, 0] = 2.0
    d2R[2, 2, 0, 0] = 2.0

    # Declare the Hessian of covariance matrix V
    d2V = np_autograd.zeros((d, d, k, k))
    d2V[0, 0, 0, 0] = 2 * sgm**2 * (1 + 3 * a**2) / (1 - a**2)**3
    d2V[1, 1, 0, 0] = 2 / (1 - a**2)
    d2V[0, 1, 0, 0] = 4 * a * sgm / (1 - a**2)**2
    d2V[1, 0, 0, 0] = 4 * a * sgm / (1 - a**2)**2

    # Initialize the x and dx
    x = np_autograd.zeros((k, 1))
    dx = np_autograd.zeros((d, 1, k, 1))
    d2x = np_autograd.zeros((d, d, k, 1))

    return F, G, H, Q, R, dF, dG, dH, dQ, dR, d2F, d2G, d2H, d2Q, d2R, V, dV, d2V, x, dx, d2x, k, d, m

In [ ]:
def log_densities_ARMA(theta, y, model_name):
    """
    Compute the log-density of the ARMA(1, 1) model
    """
    # Initialize the parameters
    if model_name == "ARMA11":
        c = 0
        F, G, H, Q, R, dF, dG, dH, dQ, dR, d2F, d2G, d2H, d2Q, d2R, V, dV, d2V, x, dx, d2x, k, d, m = initiate_KF_parameter_values_ARMA11(theta)
    elif model_name == "Vasicek_Demeaned":
        c = 0
        F, G, H, Q, R, dF, dG, dH, dQ, dR, d2F, d2G, d2H, d2Q, d2R, V, dV, d2V, x, dx, d2x, k, d, m = initiate_KF_parameter_values_Vasicek_Demeaned(theta)
    elif model_name == "Vasicek":
        c = theta[3]
        F, G, H, Q, R, dF, dG, dH, dQ, dR, d2F, d2G, d2H, d2Q, d2R, V, dV, d2V, x, dx, d2x, k, d, m = initiate_KF_parameter_values_Vasicek(theta)

    N = len(y)
    
    l_t = np_autograd.zeros(N)

    # Run the Kalman filter for gradient
    for t in range(N):
        
        # 1. PREDICT
        # Predict one-ste p-ahead state predictive density of x_{t}
        x_p = F @ x + c # Add c for Vasicek model
        V_p = F @ V @ F.T + G * Q @ G.T

        # Compute forecast error and one-step-ahead predictive variance
        e = y[t] - (H @ x_p)[0, 0]
        r = (H @ V_p @ H.T + R)[0, 0]

        # 2. UPDATE
        # Kalman gain
        K = V_p @ H.T / r

        # Update current state and covariance
        x = x_p + K * e
        V = (np_autograd.eye(k) - K @ H) @ V_p
        
        # Compute the gradient of the log-likelihood
        log_likelihood_t = -0.5 * (
            np_autograd.log(2 * np_autograd.pi) 
            + np_autograd.log(r) 
            + e**2 / r
        )

        # Store the result
        l_t[t] = log_likelihood_t
        
    # Return the dictionary
    return l_t

def log_densities_ARMA_for_numerical_Hess(theta, y, model_name):
    """
    Like log_densities_ARMA11 but returns only last element.
    """
    return log_densities_ARMA(theta, y, model_name)[-1]

def grad_log_densities_ARMA(theta, y, model_name, h, hinv_p):
    """
    Compute the gradient of the log-density of the ARMA(1, 1) model
    """

    phi = h(theta)
    hinv_p_vals = hinv_p(phi)

    # Initialize the parameters
    if model_name == "ARMA11":
        F, G, H, Q, R, dF, dG, dH, dQ, dR, d2F, d2G, d2H, d2Q, d2R, V, dV, d2V, x, dx, d2x, k, d, m = initiate_KF_parameter_values_ARMA11(theta)
        c = 0
        dC = np_autograd.zeros((d, 1, 1, 1))
    elif model_name == "Vasicek_Demeaned":
        F, G, H, Q, R, dF, dG, dH, dQ, dR, d2F, d2G, d2H, d2Q, d2R, V, dV, d2V, x, dx, d2x, k, d, m = initiate_KF_parameter_values_Vasicek_Demeaned(theta)
        c = 0
        dC = np_autograd.zeros((d, 1, 1, 1))
    elif model_name == "Vasicek":
        F, G, H, Q, R, dF, dG, dH, dQ, dR, d2F, d2G, d2H, d2Q, d2R, V, dV, d2V, x, dx, d2x, k, d, m = initiate_KF_parameter_values_Vasicek(theta)
        c = theta[3]
        dC = np_autograd.zeros((d, 1, 1, 1))
        dC[3, 0, 0, 0] = 1

    N = len(y)

    grad_l_t = np_autograd.zeros((N, d))

    # Run the Kalman filter for gradient
    for t in range(N):
        
        """
        1. PREDICT
        """

        # Predict one-step-ahead state predictive density of x_{t}
        x_p = F @ x + c # Add c for Vasicek model
        V_p = F @ V @ F.T + G @ Q @ G.T

        # Compute forecast error and one-step-ahead predictive variance
        e = y[t] - (H @ x_p)[0, 0]
        r = (H @ V_p @ H.T + R)[0, 0]  # General: Add +R

        # Predict the gradient and Hessian of x_{t} and V_{t}
        dx_p = F @ dx + dF @ x + dC # Add dC for Vasicek model
        dV_p = F @ dV @ F.T + dF @ V @ F.T + F @ V @ dF + G @ dQ @ G.T + dG @ Q @ G.T + G @ Q @ dG.swapaxes(2, 3)

        # Calculate gradient and Hessian of e and r
        de = -H @ dx_p - dH @ x_p # General: - dH @ x_p
        dr = H @ dV_p @ H.T + dH @ V_p @ H.T + H @ V_p @ dH.swapaxes(2, 3) + dR  # General: + dH @ V_p @ H.T + H @ V_p @ dH.swapaxes(2, 3) + dR           

        """
        2. UPDATE
        """

        # Kalman gain
        K = V_p @ H.T / r
        dK = (dV_p @ H.T + V_p @ dH.swapaxes(2, 3)) / r - (V_p @ H.T / r**2) @ dr  # General : + V_p @ dH.swapaxes(2, 3)

        # Update current state and covariance
        x = x_p + K * e
        dx = dx_p + K @ de + dK * e
        V = (np_autograd.eye(k) - K @ H) @ V_p
        dV = dV_p - dK @ H @ V_p - K @ dH @ V_p - K @ H @ dV_p # General: - K @ dH @ V_p
        
        # Compute the log-likelihood at the observation t
        log_likelihood_t = -0.5 * (
            np_autograd.log(2 * np_autograd.pi) 
            + np_autograd.log(r) 
            + e**2 / r
        )
        
        # Compute the gradient of the log-likelihood at the observation t
        analytical_grad_t_theta = - 0.5 * (
            1/r * dr
            + 2 * e * de / r
            - e**2 * dr / r**2
        )
        
        analytical_grad_t_phi = analytical_grad_t_theta.flatten() * hinv_p_vals

        # Store the result
        grad_l_t[t] = analytical_grad_t_phi  # .flatten()
        
    # Return the result
    return grad_l_t

def Hess_log_densities_ARMA(theta, y, model_name, h, hinv_p, hinv_pp):
    """
    Compute the Hessian of the log-density of the ARMA(1, 1) model
    """
    
    phi = h(theta)  # Convert theta -> phi
    hinv_p_vals = hinv_p(phi)  # dθ/dφ
    hinv_pp_vals = hinv_pp(phi) 


    # Initialize the parameters
    if model_name == "ARMA11":
        F, G, H, Q, R, dF, dG, dH, dQ, dR, d2F, d2G, d2H, d2Q, d2R, V, dV, d2V, x, dx, d2x, k, d, m = initiate_KF_parameter_values_ARMA11(theta)
        c = 0
        dC = np_autograd.zeros((d, 1, 1, 1))
    elif model_name == "Vasicek_Demeaned":
        F, G, H, Q, R, dF, dG, dH, dQ, dR, d2F, d2G, d2H, d2Q, d2R, V, dV, d2V, x, dx, d2x, k, d, m = initiate_KF_parameter_values_Vasicek_Demeaned(theta)
        c = 0
        dC = np_autograd.zeros((d, 1, 1, 1))
    elif model_name == "Vasicek":
        F, G, H, Q, R, dF, dG, dH, dQ, dR, d2F, d2G, d2H, d2Q, d2R, V, dV, d2V, x, dx, d2x, k, d, m = initiate_KF_parameter_values_Vasicek(theta)
        c = theta[3]
        dC = np_autograd.zeros((d, 1, 1, 1))
        dC[3, 0, 0, 0] = 1

    N = len(y)

    Hess_l_t = np_autograd.zeros((N, d, d))

    # Run the Kalman filter for gradient
    # Run the Kalman filter for gradient
    for t in range(N):
        
        """
        1. PREDICT
        """

        # Predict one-step-ahead state predictive density of x_{t}
        x_p = F @ x + c # Add c for Vasicek model
        V_p = F @ V @ F.T + G @ Q @ G.T

        # Compute forecast error and one-step-ahead predictive variance
        e = y[t] - (H @ x_p)[0, 0]
        r = (H @ V_p @ H.T + R)[0, 0]  # General: Add +R

        # Predict the gradient and Hessian of x_{t} and V_{t}
        dx_p = F @ dx + dF @ x + dC # Add dC for Vasicek model
        dV_p = F @ dV @ F.T + dF @ V @ F.T + F @ V @ dF + G @ dQ @ G.T + dG @ Q @ G.T + G @ Q @ dG.swapaxes(2, 3)
        d2x_p = (dF @ dx.swapaxes(0, 1)) + (dF @ dx.swapaxes(0, 1)).swapaxes(0, 1) \
                    + F @ d2x \
                    + d2F @ x
        d2V_p = (dF @ dV.swapaxes(0, 1) @ F.T) + (dF @ dV.swapaxes(0, 1) @ F.T).swapaxes(0, 1) \
                    + F @ d2V @ F.T \
                    + (F @ dV @ dF.transpose(1, 0, 3, 2)) + (F @ dV @ dF.transpose(1, 0, 3, 2)).swapaxes(0, 1) \
                    + d2F @ V @ F.T \
                    + (dF @ V @ dF.transpose(1, 0, 3, 2)) + (dF @ V @ dF.transpose(1, 0, 3, 2)).swapaxes(0, 1) \
                    + F @ V @ d2F.swapaxes(2, 3) \
                    + (dG @ dQ.swapaxes(0, 1) @ G.T) + (dG @ dQ.swapaxes(0, 1) @ G.T).swapaxes(0, 1) \
                    + G @ d2Q @ G.T \
                    + (G @ dQ @ dG.transpose(1, 0, 3, 2)) + (G @ dQ @ dG.transpose(1, 0, 3, 2)).swapaxes(0, 1) \
                    + d2G @ Q @ G.T \
                    + (dG @ Q @ dG.transpose(1, 0, 3, 2)) + (dG @ Q @ dG.transpose(1, 0, 3, 2)).swapaxes(0, 1) \
                    + G @ Q @ d2G.swapaxes(2, 3) 

        # Calculate gradient and Hessian of e and r
        de = -H @ dx_p - dH @ x_p # General: - dH @ x_p
        dr = H @ dV_p @ H.T + dH @ V_p @ H.T + H @ V_p @ dH.swapaxes(2, 3) + dR  # General: + dH @ V_p @ H.T + H @ V_p @ dH.swapaxes(2, 3) + dR
        d2e = - (dH @ dx_p.swapaxes(0, 1)) - (dH @ dx_p.swapaxes(0, 1)).swapaxes(0, 1) \
                - H @ d2x_p \
                - d2H @ x_p
        d2r = (dH @ dV_p.swapaxes(0, 1) @ H.T) + (dH @ dV_p.swapaxes(0, 1) @ H.T).swapaxes(0, 1) \
                + H @ d2V_p @ H.T \
                + (H @ dV_p @ dH.transpose(1, 0, 3, 2)) + (H @ dV_p @ dH.transpose(1, 0, 3, 2)).swapaxes(0, 1) \
                + d2H @ V_p @ H.T \
                + (dH @ V_p @ dH.transpose(1, 0, 3, 2)) + (dH @ V_p @ dH.transpose(1, 0, 3, 2)).swapaxes(0, 1) \
                + H @ V_p @ d2H.swapaxes(2, 3) \
                + d2R # General: + d2R
                

        """
        2. UPDATE
        """

        # Kalman gain
        K = V_p @ H.T / r
        dK = (dV_p @ H.T + V_p @ dH.swapaxes(2, 3)) / r - (V_p @ H.T / r**2) @ dr  # General : + V_p @ dH.swapaxes(2, 3)
        d2K = (d2V_p @ H.T 
            + (dV_p @ dH.transpose(1, 0, 3, 2)) + (dV_p @ dH.transpose(1, 0, 3, 2)).swapaxes(0, 1)
            + V_p @ d2H.swapaxes(2, 3)) * (1/r) \
            - (dV_p @ H.T + V_p @ dH.swapaxes(2, 3)) * (1/r**2) @ dr.swapaxes(0, 1) \
            - ((dV_p @ H.T + V_p @ dH.swapaxes(2, 3)) * (1/r**2) @ dr.swapaxes(0, 1)).swapaxes(0, 1) \
            + (V_p @ H.T) * (1/r**3) @ dr @ dr.swapaxes(0, 1) + ((V_p @ H.T) * (1/r**3) @ dr @ dr.swapaxes(0, 1)).swapaxes(0, 1) \
            - V_p @ H.T * (1/r**2) @ d2r 

        # Update current state and covariance
        x = x_p + K * e
        dx = dx_p + K @ de + dK * e
        d2x = d2x_p \
            + (dK @ de.swapaxes(0, 1)) + (dK @ de.swapaxes(0, 1)).swapaxes(0, 1) \
            + K @ d2e \
            + d2K * e

        V = (np_autograd.eye(k) - K @ H) @ V_p
        dV = dV_p - dK @ H @ V_p - K @ dH @ V_p - K @ H @ dV_p # General: - K @ dH @ V_p
        d2V = d2V_p \
            - d2K @ H @ V_p \
            - (dK @ dH.swapaxes(0, 1) @ V_p) - (dK @ dH.swapaxes(0, 1) @ V_p).swapaxes(0, 1) \
            - (dK @ H @ dV_p.swapaxes(0, 1)) - (dK @ H @ dV_p.swapaxes(0, 1)).swapaxes(0, 1) \
            - K @ d2H @ V_p \
            - (K @ dH @ dV_p.swapaxes(0, 1)) - (K @ dH @ dV_p.swapaxes(0, 1)).swapaxes(0, 1) \
            - K @ H @ d2V_p 
        
        # Compute the log-likelihood at the observation t
        log_likelihood_t = -0.5 * (
            np_autograd.log(2 * np_autograd.pi) 
            + np_autograd.log(r) 
            + e**2 / r
        )
        
        # Compute the gradient of the log-likelihood at the observation t
        analytical_grad_t_theta = - 0.5 * (
            1/r * dr
            + 2 * e * de / r
            - e**2 * dr / r**2
        )
        
        # Compute the Hessian of the log-likelihood at the observation t
        analytical_Hess_t_theta = -0.5 * (
            (1/r) * (1 - (e**2 / r)) * d2r 
            + 2 * (e/r) * d2e 
            + ((2 * e**2 / r**3) - (1/r**2)) * dr @ dr.swapaxes(0, 1)
            + (2 / r) * de @ de.swapaxes(0, 1)
            - 2 * (e / r**2) * (de @ dr.swapaxes(0, 1) + dr @ de.swapaxes(0, 1))
        )


        grad_theta_vec = analytical_grad_t_theta.flatten()
        H_theta_mat = analytical_Hess_t_theta.squeeze()

        J = hinv_p_vals.flatten()
        J2 = hinv_pp_vals.flatten()

        term1 = np_autograd.diag(J) @ H_theta_mat @ np_autograd.diag(J)
        # term1 = np.outer(hinv_p_vals[:d_no_v], hinv_p_vals[:d_no_v])*Hess_term
        term2 = np_autograd.diag(grad_theta_vec * J2)

        analytical_Hess_t_phi = term1 + term2

        Hess_l_t[t, :, :] = analytical_Hess_t_phi
        
    # Return the dictionary
    return Hess_l_t

def initiate_model_functions(model_name, h, hinv, hinv_p, hinv_pp):
    
    def log_densities(theta, y, model_name):
        return log_densities_ARMA(theta, y, model_name)
        
    def grad_log_densities(theta, y, model_name):
        """
        Gradient with respect to phi, but input is theta (theta = hinv(phi)).
        """
        return grad_log_densities_ARMA(theta, y, model_name, h=h, hinv_p=hinv_p)

    def Hess_log_densities(theta, y, model_name):
        """
        Gradient with respect to phi, but input is theta (theta = hinv(phi)).
        """
        return Hess_log_densities_ARMA(theta, y, model_name, h=h, hinv_p=hinv_p, hinv_pp=hinv_pp)
    
    return log_densities, grad_log_densities, Hess_log_densities


"""
Define the corresponding log-likelihood functions
"""
def log_likelihood(theta, y, model_name):
    """
    The log-likelihood for the full sample.
    Function of original parameters theta.
    """
    return np_autograd.sum(log_densities(theta, y, model_name))

def grad_log_likelihood(theta, y, model_name):
    """
    Gradient of the log-likelihood for the full sample.
    Gradient with respect to phi, but input is theta (theta = hinv(phi)).
        
    """
    return np_autograd.sum(grad_log_densities(theta, y, model_name), axis = 0)

def Hess_log_likelihood(theta, y, model_name):
    """
    Hessian of the log-likelihood for the full sample.
    Hessian with respect to phi, but input is theta (theta = hinv(phi)).
    """
    return np_autograd.sum(Hess_log_densities(theta, y, model_name), axis = 0)


"""
Define the prior and posterior. Note, the posterior is with respect to phi, where phi is the unrestricted parameter.
Prior is defined on theta (original parameterisation), so we need the determinant of the Jacobian.
"""
def log_prior_ARMA11(theta):
    """
    Simple indepent normal prior with mean 0 and variance 10^2
    """
    a1, b1, sigma = theta[0], theta[1], theta[2]
    lp_a1 = sps_autograd.norm.logpdf(a1, 0.0, 10.0)
    lp_b1 = sps_autograd.norm.logpdf(b1, 0.0, 10.0)
    lp_sigma = np_autograd.where(sigma > 0.0,
                                 sps_autograd.norm.logpdf(sigma, 0.0, 10.0),
                                 -np_autograd.inf)
    return lp_a1 + lp_b1 + lp_sigma

def log_prior_Vasicek(theta):
    """
    Simple indepent normal prior with mean 0 and variance 10^2
    """
    a1, sgm, tau, mu = theta[0], theta[1], theta[2], theta[3]
    lp_a1 = sps_autograd.norm.logpdf(a1, 0.0, 10.0)
    lp_sgm = np_autograd.where(sgm > 0.0,
                                 sps_autograd.norm.logpdf(sgm, 0.0, 10.0),
                                 -np_autograd.inf)
    lp_tau = np_autograd.where(tau > 0.0,
                                 sps_autograd.norm.logpdf(tau, 0.0, 10.0),
                                 -np_autograd.inf)
    lp_mu = sps_autograd.norm.logpdf(mu, 0.0, 10.0)
    return lp_a1 + lp_sgm + lp_tau + lp_mu

def log_prior_Vasicek_Demeaned(theta):
    """
    Simple indepent normal prior with mean 0 and variance 10^2
    """
    a1, sgm, tau = theta[0], theta[1], theta[2]
    lp_a1 = sps_autograd.norm.logpdf(a1, 0.0, 10.0)
    lp_sgm = np_autograd.where(sgm > 0.0,
                                 sps_autograd.norm.logpdf(sgm, 0.0, 10.0),
                                 -np_autograd.inf)
    lp_tau = np_autograd.where(tau > 0.0,
                                 sps_autograd.norm.logpdf(tau, 0.0, 10.0),
                                 -np_autograd.inf)
    return lp_a1 + lp_sgm + lp_tau

def log_penalty_ARMA11(
    theta,
    lambda_pos=10.0,
    lambda_stat=100.0, delta_stat=0.02):
    """
    Pure constraint penalty for ARMA(1,1).

    theta = [a_1, b_1, sigma]

    - No size / shrinkage terms.
    - Penalty = 0 when all constraints are satisfied.
    - Quadratic growth when constraints are violated.
    """

    # unpack
    a = theta[0]  # AR coefficient
    b = theta[1]  # MA coefficient
    sigma = theta[2]  # standard deviation

    # ---------- 1) Positivity constraint for sigma ----------
    neg_sigma = np_autograd.maximum(0.0, -sigma)
    pen_pos = lambda_pos * neg_sigma**2

    # ---------- 2) Stationarity: |a| < 1 - delta_stat ----------
    viol_ar = np_autograd.maximum(0.0, np_autograd.abs(a) - (1.0 - delta_stat))
    pen_stat_ar = lambda_stat * viol_ar**2

    # ---------- 3) Invertibility: |b| < 1 - delta_stat ----------
    viol_ma = np_autograd.maximum(0.0, np_autograd.abs(b) - (1.0 - delta_stat))
    pen_stat_ma = lambda_stat * viol_ma**2

    # total constraint penalty
    penalty = pen_pos + pen_stat_ar + pen_stat_ma
    return penalty

def log_penalty_Vasicek(
    theta,
    lambda_pos=10.0,
    lambda_stat=100.0, delta_stat=0.02):
    """
    Pure constraint penalty for Vasicek.

    theta = [a_1, b_1, sigma]

    - No size / shrinkage terms.
    - Penalty = 0 when all constraints are satisfied.
    - Quadratic growth when constraints are violated.
    """

    # unpack
    a = theta[0]  # AR coefficient
    sgm = theta[1]  # MA coefficient
    tau = theta[2]  # standard deviation
    mu = theta[3]  # standard deviation

    # ---------- 1) Positivity constraint for sigma ----------
    neg_sgm = np_autograd.maximum(0.0, -sgm)
    pen_pos = lambda_pos * neg_sgm**2

    # ---------- 2) Stationarity: |a| < 1 - delta_stat ----------
    viol_a = np_autograd.maximum(0.0, np_autograd.abs(a) - (1.0 - delta_stat))
    pen_stat_a = lambda_stat * viol_a**2

    # ---------- 3) Positivity constraint for sigma ----------
    viol_tau = np_autograd.maximum(0.0, np_autograd.abs(tau) - (1.0 - delta_stat))
    pen_stat_tau = lambda_stat * viol_tau**2

    # total constraint penalty
    penalty = pen_pos + pen_stat_a + pen_stat_tau
    return penalty

def log_penalty_Vasicek_Demeaned(
    theta,
    lambda_pos=10.0,
    lambda_stat=100.0, delta_stat=0.02):
    """
    Pure constraint penalty for Vasicek.

    theta = [a_1, b_1, sigma]

    - No size / shrinkage terms.
    - Penalty = 0 when all constraints are satisfied.
    - Quadratic growth when constraints are violated.
    """

    # unpack
    a = theta[0]  # AR coefficient
    sgm = theta[1]  # MA coefficient
    tau = theta[2]  # standard deviation

    # ---------- 1) Positivity constraint for sigma ----------
    neg_sgm = np_autograd.maximum(0.0, -sgm)
    pen_pos = lambda_pos * neg_sgm**2

    # ---------- 2) Stationarity: |a| < 1 - delta_stat ----------
    viol_a = np_autograd.maximum(0.0, np_autograd.abs(a) - (1.0 - delta_stat))
    pen_stat_a = lambda_stat * viol_a**2

    # ---------- 3) Positivity constraint for sigma ----------
    viol_tau = np_autograd.maximum(0.0, np_autograd.abs(tau) - (1.0 - delta_stat))
    pen_stat_tau = lambda_stat * viol_tau**2

    # total constraint penalty
    penalty = pen_pos + pen_stat_a + pen_stat_tau
    return penalty

def log_posterior(phi, y, model_name):
    """
    The log posterior is parameterised in the phi space (log_prior_phi includes the log-determinant of the Jacobian) 
    Note that log_prior_phi will be created when a prior for the model is initiated, by calling the initiate_priors function.
    """
    theta = hinv(phi)

    if model_name == "ARMA11":
        lprior = log_prior_ARMA11(theta)
    elif model_name == "Vasicek":
        lprior = log_prior_Vasicek(theta)
    elif model_name == "Vasicek_Demeaned":
        lprior = log_prior_Vasicek_Demeaned(theta)
    
    if np_autograd.isinf(lprior):
        # print("Outside of safe regime. Prior set to 0.")
        violate_prior = True
        return -np_autograd.Inf, violate_prior  
    else:
        violate_prior = False
        # Add logdet Jacobian (to get prior on phi)
        return log_likelihood(theta, y, model_name) + lprior + logdet(phi), violate_prior

def initiate_priors(model_name, error_type='normal'):    
    def log_prior_theta(theta):
        if model_name == "ARMA11":
            return log_prior_ARMA11(theta)
        elif model_name == "Vasicek":
            return log_prior_Vasicek(theta)
        elif model_name == "Vasicek_Demeaned":
            return log_prior_Vasicek_Demeaned(theta)
            
    return log_prior_theta

def initiate_penalties(model_name, error_type='normal'):    
    def log_penalty_theta(theta):
        if model_name == "ARMA11":
            return log_penalty_ARMA11(theta)
        elif model_name == "Vasicek":
            return log_penalty_Vasicek(theta)
        elif model_name == "Vasicek_Demeaned":
            return log_penalty_Vasicek_Demeaned(theta)
            
    return log_penalty_theta

### 1.4 Methodology functions
Contains functions specific for our methodology, such as those needed for the estimator (including control variates and sampling schemes), and functions for the tuning procedure.

In [ ]:
"""
Functions for control variates and estimator (including sampling probabilities)
"""
def initiate_control_variate_quantities(log_density, grad_log_density, Hess_log_density, phiStar, args):
    """
    Creates the quantities needed to construct the second order parameter expanded Taylor control variates for the log_density.
    Output from this function will go into the function eval_q_k.
    The control variate acts on phi = h(theta). Recall that grad and Hess are with respect to phi but has argument theta.
    """
    thetaStar = hinv(phiStar)
    dens_at_phiStar = log_density(thetaStar, *args) 
    grad_at_phiStar = grad_log_density(thetaStar, *args)
    Hess_at_phiStar = Hess_log_density(thetaStar, *args)
                    
    return dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar

def eval_q_k(phi, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 2):
    """
    Evaluates the order order parameter expanded Taylor control variates at the point phi for all observations in 
    dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar. Default order is 2.
    """
    const_term = dens_at_phiStar
    if order == 0:
        q_k = const_term
    elif order == 1:
        first_term = np.sum(grad_at_phiStar*(phi - phiStar), axis = 1)
        q_k = const_term + first_term
    elif order == 2:
        first_term = np.sum(grad_at_phiStar*(phi - phiStar), axis = 1)
        second_term = 0.5*np.sum(np.sum(Hess_at_phiStar*np.outer(phi - phiStar, phi - phiStar), axis = 1), axis = 1)    
        q_k = const_term + first_term + second_term
    else:
        raise ValueError("Order must be 0 <= order <= 2")
    return q_k

def TPD_probs(T, t_star, gamma, b = 100):
    """
    Truncated power-law decay (TPD) sampling probabilities.

    Parameters
    ----------
    T : int
        Support size; indices are t = 1, ..., T.
    t_star : int
        Head cut-off; 1 <= t_star < T. Head = {1, ..., t_star}, tail = {t_star+1, ..., T}.
    epsilon : float
        Total tail mass, must be in (0, 1). Head mass is (1 - epsilon).
    gamma : float, default 0.6
        Power-law exponent; may be 0 (flat head).
    b : float, default 0.0
        Non-negative offset (design parameter).

    Returns
    -------
    p : np.ndarray of shape (T,)
        Probabilities with p[0] corresponding to t=1, ..., p[T-1] to t=T.
    """
     # Validation
    if not (isinstance(T, int) and isinstance(t_star, int)):
        raise TypeError("T and t_star must be integers.")
    if not (1 <= t_star < T):
        raise ValueError("Require 1 <= t_star < T.")
    if gamma < 0:
        raise ValueError("gamma must be >= 0.")
    if b < 0:
        raise ValueError("b must be >= 0.")

    tail_n_obs = T - t_star  # tail length (>0)

    # Head weights and normaliser
    t_head = np.arange(1, t_star + 1, dtype=float)
    w = (t_head + b) ** (-gamma) if gamma != 0 else np.ones_like(t_head)
    W = w.sum()
    if not np.isfinite(W) or W <= 0:
        raise FloatingPointError("Head normalisation failed (non-finite or non-positive sum).")

    # ε(γ) from “no jump at t*”: p_{t*} (head) = p_{t*+1} (tail)
    last_head = (t_star + b) ** (-gamma) if gamma != 0 else 1.0
    denom = last_head * tail_n_obs + W
    eps = (last_head * tail_n_obs) / denom  # ε(γ)

    # Assemble probabilities
    p = np.empty(T, dtype=float)
    p[:t_star] = (1.0 - eps) * (w / W)
    p[t_star:] = eps / tail_n_obs

    # Numerical tidy-up
    p /= p.sum()
    return p, float(eps)


def variance_WDE(e_t, p_t, m):
    e = np.sum(e_t)
    return (1/m) * np.sum(((e_t / p_t) - e)**2 * p_t)

def grad_obs_variance_WDE(phi, phiStar, p_t, log_dens_func, grad_log_dens_func, Hess_log_dens_func, args):
    """
    V[l_hat] = 1/m sum v_t, where v_t is the variance contribution of the t:th observation and is a function of 
    phi (through both the log density and the control variate). This function returns v_t for all obs
    """    
    dens_at_phiStar,  grad_at_phiStar, Hess_at_phiStar = initiate_control_variate_quantities(log_dens_func, grad_log_dens_func, Hess_log_dens_func, phiStar, args)
    q_t = eval_q_k(phi, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 2)
    e_t = log_dens_func(hinv(phi), *args) - q_t    
    grad_q = grad_at_phiStar + np.dot(Hess_at_phiStar, phi - phiStar)
    grad_e = grad_log_dens_func(hinv(phi), *args) - grad_q
    
    return 2*(e_t/p_t - np.sum(e_t))[:, np.newaxis]*(grad_e - p_t[:, np.newaxis]*np.sum(grad_e, axis = 0))

def grad_variance_WDE(phi, phiStar, p_t, log_dens_func, grad_log_dens_func, Hess_log_dens_func, args, m):
    """
    Computes the gradient of the variance of the WDE estimator with respect to phi
    """
    grad_terms = grad_obs_variance_WDE(phi, phiStar, p_t, log_dens_func, grad_log_dens_func, Hess_log_dens_func, args)
    return np.sum(grad_terms, axis = 0)/m

def log_est_WDE(e_t, p_t, m):
    """
    Computes the WDE estimator of the log-likelihood
    """
    return np.mean(e_t/p_t)

def variance_hat_WDE(e_t_sub, p_t_sub, m):
    """
    unbiased estimator of variance of WDE
    """
    return np.var(e_t_sub/p_t_sub, ddof=1)/m

def expected_umax_safe(p_t, m):
    """
    Overflow-safe computation of E[u_max].

    Parameters
    ----------
    p_t : array-like
          Probabilities for t = 1,...,T (must sum to 1).
    m : int
        Subsample size. 

    Returns
    -------
    float
        Expected maximum index u_max.
    """
    T = len(p_t)
    # cumulative sums from p_1 up to p_{k-1}
    cum_probs = np.cumsum(p_t)
    terms = np.concatenate([[0.0], cum_probs[:-1]])  # shift so t=1 term is 0

    # Clip terms to avoid log(0) or log(>1)
    terms_clipped = np.clip(terms, 1e-300, 1.0)  

    # Compute (terms)^m using exp(m * log(terms))
    log_terms = np.log(terms_clipped)
    powers = np.exp(m * log_terms)

    return T - np.sum(powers)


def eps_exact(T, t_star, gamma, b):
    """
    Used to compute the numerical solution to gamma_max.  
    """
    t_vals = np.arange(1, t_star + 1, dtype=float)
    w = (t_vals + b) ** (-gamma)
    W = w.sum()
    last_head = (t_star + b) ** (-gamma)
    return (last_head * (T - t_star)) / (last_head * (T - t_star) + W)


def gamma_max_value(T, t_star, c, b, tol=1e-10, max_iter=100):
    """
    The tail mass as a function of gamma satisfies:
    
    eps(gamma) >= c(T - t_star)/T. eps(gamma) decreases in gamma
    
    This function computes the largest gamma such that the the above hold, i.e.
    eps(gamma_max) = c(T - t_star)/T . For any 0 < gamma < gamma_max, eps(gamma) >= eps(gamma_max)
    
    """
    target = c * (T - t_star) / T

    def eps(g):
        return eps_exact(T, t_star, g, b)

    # bracket for exact gamma_max
    lo, hi = 0.0, 1.0
    while eps(hi) > target and hi < 1e6:
        hi *= 2.0
    gamma_max = brentq(lambda g: eps(g) - target, lo, hi, xtol=tol, maxiter=max_iter)

    return float(gamma_max)

"""
Functions for tuning using Approach 1 in the paper: Minimise variance for a given expected compute
"""

def m_star(c, alpha, T, t_star, b):
    """
    m_star is the largest subsample (given alpha) such that E(u_max; c, m) is 
    less than the compute budget alpha*T. Recall that such a subsample may not exist for all c, but we ensure 
    we evaluate over the feasible set.
    """
    gamma_max = gamma_max_value(T, t_star, c, b)                        
    probs, eps = TPD_probs(T, t_star, gamma = gamma_max, b = b)     
    thr = alpha * T
    m = 1
    m_max = T # int(T/2)
    val = expected_umax_safe(probs, m)
    if val > alpha * T:
        raise ValueError("c = %s is infeasible" % c)
    while val <= thr and m < m_max:
        m += 1
        #print(c)
        #print(m)
        val = expected_umax_safe(probs, m)
    return m - 1  # largest m with E <= alpha*T    


def m_star_gamma(gamma, alpha, T, t_star, b):
    """
    NOTE: This was just to check what happens if we optimise (gamma, m) rather than (c, m). Gives essentially the same result.
    
    m_star is the largest subsample (given alpha) such that E(u_max; gamma, m) is 
    less than the compute budget alpha*T. Recall that such a subsample may not exist for all gamma, but we ensure 
    we evaluate over the feasible set.
    """                        
    probs, eps = TPD_probs(T, t_star, gamma = gamma, b = b)     
    thr = alpha * T
    m = 1
    m_max = T # int(T/2)
    val = expected_umax_safe(probs, m)
    if val >= alpha * T:
        raise ValueError("gamma is infeasible")
    while val <= thr and m < m_max:
        m += 1
        #print(gamma)
        #print(m)
        val = expected_umax_safe(probs, m)

    return m - 1  # largest m with E <= alpha*T    

def objective_1_minimise_var(c, e_t, T, t_star, b, alpha):                      
    """
    Objective 1: Minimise variance (which includes e_t) given an expected compute budget.
    """
    gamma_max = gamma_max_value(T, t_star, c, b)                        
    p_t, eps = TPD_probs(T, t_star, gamma = gamma_max, b = b)     
    e = np.sum(e_t)
    sigma2_c = np.sum(((e_t / p_t) - e)**2 * p_t)
    m_alpha = m_star(c, alpha, T, t_star, b)
    #print("1/(c*m_alpha)", 1/(c*m_alpha))
    #print("sigma2_c/(m_alpha)", sigma2_c/(m_alpha))
    #print("ratio", (sigma2_c/(m_alpha))/ (1/(c*m_alpha)))
    return sigma2_c/(m_alpha)


def objective_1_minimise_var_upper_bound(c, T, t_star, b, alpha):                      
    """
    Objective 1: Minimise upper bound of variance (which does not include e_t) given an expected compute budget.
    """
    gamma_max = gamma_max_value(T, t_star, c, b)                        
    p_t, eps = TPD_probs(T, t_star, gamma = gamma_max, b = b)     
    m_alpha = m_star(c, alpha, T, t_star, b)
    return 1/(c*m_alpha)


def objective_1_minimise_var_gamma(gamma, e_t, T, t_star, b, alpha):                      
    """
    NOTE: This was just to check what happens if we optimise (gamma, m) rather than (c, m). Gives essentially the same result.
    """
    p_t, eps = TPD_probs(T, t_star, gamma = gamma, b = b)     
    e = np.sum(e_t)
    sigma2_c = np.sum(((e_t / p_t) - e)**2 * p_t)
    m_alpha = m_star_gamma(gamma, alpha, T, t_star, b)
    return sigma2_c/m_alpha

def defensive_minimize_scalar(f, a, b, args, n_grid=1001, pad=0.05, xatol=1e-10, maxiter=500):
    """
    minimize_scalar was not finding the best solution, so I asked chatGPT for advise and I got this
    defensive minimize scalar strategy.
    """
    # 1) coarse scan
    xs = np.linspace(a, b, n_grid)
    fs = np.array([f(x, *args) for x in xs])
    k = int(np.argmin(fs))
    x_star0 = xs[k]

    # 2) derive a tight local window around the best grid point
    width = (b - a)
    left  = max(a, x_star0 - pad*width)
    right = min(b, x_star0 + pad*width)
    if right <= left:  # fallback
        left, right = a, b

    # 3) bounded polish in the window (robust)
    res_local = minimize_scalar(
        f, args = args, method="bounded", bounds=(left, right),
        options={"xatol": xatol, "maxiter": maxiter}
    )

    # 4) optional global polish over full bounds (useful if window was unlucky)
    res_global = minimize_scalar(
        f, args = args, method="bounded", bounds=(a, b),
        options={"xatol": xatol, "maxiter": maxiter}
    )

    # 5) pick the better of the two
    return res_local if res_local.fun <= res_global.fun else res_global


"""
Functions for tuning using Approach 2 in the paper: Minimise expected compute given a variance tolerance
"""

def c_min_feasible_E(e_t, T, t_star, b, V,
                     c_lo=1e-9, c_hi=1.0,
                     tol=1e-8, max_iter=80,
                     loose_ratio=5.0,
                     loose_abs=None,
                     pmin=1e-12):
    """
    Asked ChatGPT to write a function that finds the smallest c in (0,1] such that sigma2_c(c) <= V*T holds, assuming sigma2_c(c) is decreasing in c.
    Also warns if the constraint is too loose: even at the tightest c (~0), sigma2 << V*T.

    Find the smallest c in (c_lo, c_hi] s.t. sigma2_c(c) <= V*T.
    Assumes sigma2_c(c) is decreasing in c.
    """
    bound = V * T
    e_sum = float(np.sum(e_t))

    def sigma2_c(c):
        gamma_max = gamma_max_value(T, t_star, c, b)
        p_t, _ = TPD_probs(T, t_star, gamma=gamma_max, b=b)
        p_t = np.maximum(p_t, pmin)  # numerical safety
        # sigma^2(c) = sum_t ((e_t/p_t) - e)^2 * p_t, with e = sum_t e_t
        return float(np.sum(((e_t / p_t) - e_sum) ** 2 * p_t))

    s_lo = sigma2_c(c_lo)
    s_hi = sigma2_c(c_hi)

    warn_loose = False
    if s_lo <= bound:
        if (loose_ratio is not None and bound / max(s_lo, 1e-300) >= loose_ratio) or \
           (loose_abs is not None and (bound - s_lo) >= loose_abs):
            warn_loose = True
        return c_lo, {'sigma2_lo': s_lo, 'sigma2_hi': s_hi, 'bound': bound,
                      'warn_loose': warn_loose, 'status': 'feasible_at_c_lo'}

    if s_hi > bound:
        return None, {'sigma2_lo': s_lo, 'sigma2_hi': s_hi, 'bound': bound,
                      'warn_loose': False, 'status': 'infeasible_on_bracket'}

    lo, hi = c_lo, c_hi
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        s_mid = sigma2_c(mid)
        if s_mid <= bound:
            hi = mid
        else:
            lo = mid
        if hi - lo <= tol:
            break

    c_star = hi
    return c_star, {'sigma2_lo': s_lo, 'sigma2_hi': s_hi, 'bound': bound,
                    'warn_loose': False, 'status': 'bisection_converged'}


def m_star_V(c, V, e_t, T, t_star, b, pmin=1e-12):
    """
    Smallest integer m such that Var_hat ≈ sigma2_c(c)/m <= V.
    Returns at least 1.
    """
    gamma_max = gamma_max_value(T, t_star, c, b)
    p_t, _ = TPD_probs(T, t_star, gamma=gamma_max, b=b)
    p_t = np.maximum(p_t, pmin)
    e = float(np.sum(e_t))
    sigma2_c = float(np.sum(((e_t / p_t) - e) ** 2 * p_t))
    m_needed = int(max(1, np.ceil(sigma2_c / V)))
    return m_needed


def objective_2_minimise_E(c, e_t, T, t_star, b, V, pmin=1e-12):
    """
    Objective 2: minimise expected compute given a variance tolerance V.
    For any c, choose the minimal feasible m = m_star_V(...), then
    return E[u_max] under those (p_t, m).
    """
    gamma_max = gamma_max_value(T, t_star, c, b)
    p_t, _ = TPD_probs(T, t_star, gamma=gamma_max, b=b)
    p_t = np.maximum(p_t, pmin)
    m_V = m_star_V(c, V, e_t, T, t_star, b, pmin=pmin)
    return expected_umax_safe(p_t, m_V)

def tune_c_m_approach2(V_tolerance, e_t_tune):
    """
    Objective: Minimise expected compute given a variance tolerance.
    """
    # Step 1: Find the smallest c in (c_lo, c_hi] s.t. sigma2_c(c) <= V*T.
    #         Assumes sigma2_c(c) is decreasing in c.
    ans, info = c_min_feasible_E(e_t_tune, T, t_star, b, V_tolerance)

    if info['status'] == 'infeasible_on_bracket':
        raise RuntimeError(
            f"No feasible c in bracket: sigma2(c_hi)={info['sigma2_hi']:.3e} > bound={info['bound']:.3e}."
        )
    c_min = ans
    
    print("------------------------------------------------------------------------------")
    print("Minimise expected compute given variance tolerance                            ")
    print("------------------------------------------------------------------------------")
    opt_c_min_E = defensive_minimize_scalar(objective_2_minimise_E, c_min, 1, args = (e_t_tune, T, t_star, b, V_tolerance))
    #print('Finished defensive_minimize_scalar')
    c_opt_min_E = opt_c_min_E.x
    m_opt_min_E = m_star_V(c_opt_min_E, V_tolerance, e_t_tune_approach2, T, t_star, b)
    print("Tuning using minimise expcted compute under fixed variance V = ", V_tolerance)
    print("Optimal c = ", c_opt_min_E, "optimal m = ", m_opt_min_E, "c_min = ", c_min)
    print("Variance and expected compute under the optimal values")
    gamma_max_opt = gamma_max_value(T, t_star, c_opt_min_E, b)                        
    probs_opt, eps_opt = TPD_probs(T, t_star, gamma = gamma_max_opt, b = b)     
    print("Expected compute")
    print(expected_umax_safe(probs_opt, m_opt_min_E))
    print("Variance")
    print(variance_WDE(e_t_tune, probs_opt, m_opt_min_E))
    print("Implied gamma_max with c_opt:", gamma_max_opt)
    print("Implied eps with c_opt:", eps_opt)
    print("------------------------------------------------------------------------------")

    return c_opt_min_E, m_opt_min_E


### 1.5 Samplers and optimisers

This section contains samplers and variational optimiser.

In [ ]:
# Here goes MCMC and variational Bayes functions.
def sample_full_data_posterior_random_walk(phi_init, prop_cov, N, names, dataset_name, adapt = False):
    """
    Samples the log_posterior function using the full data likelihood. Assumes all inputs have been defined. Samples the unconstrained space 
    and returns both the constrained and unconstrained samples.
    
    If adapt = False the covariance matrix is 2.38/sqrt(d)*Posterior covariance, where posterior covariance
    is computed by inverting the negative Hessian. 
    
    Use adapt = True when the negative Hessian is not invertable (it is only positive definite in expectation). In such cases
    we estimate the covariance by the sum of the outer-product of scores. A scaled version of this is prop_cov, and the algorithm
    uses the adaption of the scale using a Robins-Monro process as in Garthwaite et al. (2016)
    
    Reference:
    Garthwaite, P. H., Fan, Y., & Sisson, S. A. (2016). Adaptive optimal scaling of Metropolis–Hastings algorithms using the Robbins–Monro process. 
    Communications in Statistics - Theory and Methods, 45(17), 5098–5111.    
    """

    # Storage
    samples_phi = np.zeros((N + 1, n_params))
    samples_phi[0, :] = phi_init
    samples_theta = np.zeros((N + 1, n_params))
    samples_theta[0, :] = hinv(phi_init)
    log_p_samples = np.zeros(N + 1)
    log_p_samples[0], crap = log_posterior(phi_init, y, model_name) # WINSTON ADJUST
    alphas_MH = np.zeros(N)
    
    d = phi_init.shape[0]                      
    target_acc = 0.23                          
    if adapt:                                  
        log_scale = np.log(0.1)               # tart with small scale (s0 = 0.1)
    else:                                     
        log_scale = 0.0                       # scale = 1.0 when not adapting (standard random walk)
    
    # Current parameter and evaluate quantities
    phi_c = phi_init      
    log_post_c, violate_prior = log_posterior(phi_c, y, model_name) # WINSTON ADJUST
    if violate_prior:
        warnings.warn("Starting value has no prior/posterior support.")
        outside_support = 1
    else:
        outside_support = 0

    tic = time.time()
    print('MCMC with random walk proposal for the %s(%s,%s) model. T = %s' % ("ARMA", 1, 1, T)) # WINSTON ADJUST
    print('Dataset name', dataset_name)
    for i in range(1, N + 1):
        if i % 100 == 0:
            print("Iteration i = {}. Acceptance prob (mean) {:.2f}. Time: {:.2f}. Outside prior/posterior support samples {:d}".format(i , np.mean(alphas_MH[:i]), time.time() - tic, outside_support))
            
        scale = np.exp(log_scale)                             
        prop_cov_eff = scale**2*prop_cov                  
        # Propose parameter and evaluate quantities
        phi_p = np.random.multivariate_normal(phi_c, prop_cov_eff, size = 1).flatten()
        log_post_p, violate_prior = log_posterior(phi_p, y, model_name) # WINSTON ADJUST
        if violate_prior:
            outside_support += 1
            
        log_q_p = sps.multivariate_normal.logpdf(phi_p, mean = phi_c, cov = prop_cov_eff)
        log_q_c = sps.multivariate_normal.logpdf(phi_c, mean = phi_p, cov = prop_cov_eff)
    
        alpha_MH = np.min([1, np.exp(log_post_p - log_q_p - (log_post_c - log_q_c))])
        alphas_MH[i - 1] = alpha_MH
        u = np.random.rand()                                 
        if u < alpha_MH: # sample Unif(0, 1) to determine acceptance
            acc_indicator = 1.0                               
            samples_phi[i, :] = phi_p
            samples_theta[i, :] = hinv(phi_p)
            log_p_samples[i] = log_post_p 
            # Proposed becomes current in next iteration
            phi_c, log_post_c, log_q_c = phi_p, log_post_p, log_q_p
        else:
            acc_indicator = 0.0                              
            samples_phi[i, :] = phi_c
            samples_theta[i, :] = hinv(phi_c)
            log_p_samples[i] = log_post_c


        if adapt:
            # Simple step size a_i = c / i; you can tune c if needed
            a_i = 1.0/i
            log_scale = log_scale + a_i*(acc_indicator - target_acc)

    run_time = time.time() - tic
    results = {"samples_theta": samples_theta, "samples_phi": samples_phi, "log_p_samples": log_p_samples,
                  "alphas_MH": alphas_MH, "outside_support": outside_support, "N": N, "run_time": run_time, "names": names, "dataset_name": dataset_name}
    return results

def sample_subsampling_posterior_random_walk(estimator_params, phi_init, prop_cov, N, names, dataset_name, adapt=False):
    """
    Samples the log_posterior function using the estimated likelihood. Assumes all inputs have been defined. Samples the unconstrained space 
    and returns both the constrained and unconstrained samples.
    
    If adapt = False the covariance matrix is 2.38/sqrt(d)*Posterior covariance, where posterior covariance
    is computed by inverting the negative Hessian. 
    
    Use adapt = True when the negative Hessian is not invertable (it is only positive definite in expectation). In such cases
    we estimate the covariance by the sum of the outer-product of scores. A scaled version of this is prop_cov, and the algorithm
    uses the adaption of the scale using a Robins-Monro process as in Garthwaite et al. (2016)
    
    Reference:
    Garthwaite, P. H., Fan, Y., & Sisson, S. A. (2016). Adaptive optimal scaling of Metropolis–Hastings algorithms using the Robbins–Monro process. 
    Communications in Statistics - Theory and Methods, 45(17), 5098–5111.    
    """

    # Read in estimator hyperparameters.
    # Sampling weights
    m = estimator_params['m_opt']
    if m == 1:
        # Subsampling MCMC requires m >= 2 (estimates the variance of the estimator unbiasedly, denominator m - 1)
        m = m + 1        
    c = estimator_params['c_opt']
    gamma = estimator_params['gamma_opt'] 
    b = estimator_params['b']
    t_star = estimator_params['t_star']
    p_t, eps = TPD_probs(T, t_star, gamma , b)    
    
    # Storage
    samples_phi = np.zeros((N + 1, n_params))
    samples_theta = np.zeros((N + 1, n_params))
    u_max_p_vals = np.zeros(N)
    log_posthat_samples = np.zeros(N + 1) # Keeps the estimated value of the log-posterior (up to a normalisation constant)
    sigma2_lhat_samples = np.zeros(N + 1) # Keeps the estimated variance of the log-likelihood estimator 
    alphas_MH = np.zeros(N)

    d = phi_init.shape[0]                  
    target_acc = 0.23                      
    if adapt:                              
        log_scale = np.log(0.1)            
    else:                                  
        log_scale = 0.0                    

    # Current u, phi and theta (starting values)
    phi_c = phi_init
    theta_c = hinv(phi_init)
    lprior_c = log_prior_ARMA11(theta_c) # WINSTON ADJUST
    
    if np_autograd.isinf(lprior_c):
        outside_support = 1
        # print("Outside of safe regime. Prior set to 0.")
        warnings.warn("Starting value has no prior/posterior support.")
        raise ValueError("Subsampling MCMC requires a valid initialisation point")
    else:
        u_c = random.choices(range(T), weights = p_t, k = m)
        u_max_c = np.max(u_c)
        l_k_u_max_c = log_densities(theta_c, y[:u_max_c + 1], model_name)
        l_k_c = l_k_u_max_c[u_c]
        q_k_c = eval_q_k(phi_c, phiStar, dens_at_phiStar[u_c], grad_at_phiStar[u_c], Hess_at_phiStar[u_c], order = 2)  # control variates at current theta, u
        log_posthat_c = q_sum(phi_c)  + log_est_WDE(l_k_c - q_k_c, p_t[u_c], m) + lprior_c + logdet(phi_c)
        sigma2_lhat_c = variance_hat_WDE(l_k_c - q_k_c, p_t[u_c], m)
        violate_prior = False
        outside_support = 0

    u_max_p = u_max_c # In case first proposed draw is outside posterior support
    print('Estimated likelihood at current')
    print(q_sum(phi_c) + log_est_WDE(l_k_c - q_k_c, p_t[u_c], m))
    print('True value at current')
    print(log_likelihood(theta_c, y, model_name)) # WINSTON ADJUST
    print('Estimated variance at current')

    print(sigma2_lhat_c)
    print('True variance at current')
    l_k_all = log_densities(theta_c, y, model_name) # WINSTON ADJUST
    q_k_all = eval_q_k(phi_c, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 2)  # control variates at current theta, phi, u
    print(variance_WDE(l_k_all - q_k_all, p_t, m))

    samples_phi[0, :] = phi_init
    samples_theta[0, :] = hinv(phi_init)
    log_posthat_samples[0] = log_posthat_c
    sigma2_lhat_samples[0] = sigma2_lhat_c
    
    tic = time.time()
    print('Subsampling MCMC with random walk proposal for the %s(%s,%s) model. T = %s, m_opt = %s,c_opt = %1.5f, gamma_opt = %1.2f' % ('ARMA', 1, 1, T, m, c, gamma))
    print('Dataset name', dataset_name)
    for i in range(1, N + 1):

        if i % 100 == 0:
            print("Iteration i = {}. Acceptance prob (mean) {:.2f}. Time: {:.2f}. Outside prior/posterior support samples {:d}".format(i , np.mean(alphas_MH[:i]), time.time() - tic, outside_support))


        scale = np.exp(log_scale)                      
        prop_cov_eff = scale**2*prop_cov           

        # Propose parameter vector and subsample
        phi_p = np.random.multivariate_normal(phi_c, prop_cov_eff, size = 1).flatten()

        # Check if feasible first
        theta_p = hinv(phi_p)
        lprior_p = log_prior_ARMA11(theta_p) # WINSTON ADJUST
    
        if np_autograd.isinf(lprior_p):
            violate_prior = True
            outside_support += 1
        else:
            u_p = random.choices(range(T), weights = p_t, k = m)
            u_max_p = np.max(u_p)
            l_k_u_max_p = log_densities(theta_p, y[:u_max_p + 1], model_name)  # WINSTON ADJUST
            l_k_p = l_k_u_max_p[u_p] # log-densities at proposed theta, u
            q_k_p = eval_q_k(phi_p, phiStar, dens_at_phiStar[u_p], grad_at_phiStar[u_p], Hess_at_phiStar[u_p], order = 2) # control variates at proposed theta, u
            log_posthat_p  = q_sum(phi_p) + log_est_WDE(l_k_p - q_k_p, p_t[u_p], m) + lprior_p + logdet(phi_p)
            sigma2_lhat_p = variance_hat_WDE(l_k_p - q_k_p, p_t[u_p], m)
            violate_prior = False

        # log proposal densities
        log_q_p = sps.multivariate_normal.logpdf(phi_p, mean = phi_c, cov = prop_cov_eff) 
        log_q_c = sps.multivariate_normal.logpdf(phi_c, mean = phi_p, cov = prop_cov_eff) 

        if not violate_prior:
            alpha_MH = np.min([1, np.exp(log_posthat_p - sigma2_lhat_p/2 - log_q_p - (log_posthat_c - sigma2_lhat_c/2 - log_q_c))])
        else:
            alpha_MH = 0

        alphas_MH[i - 1] = alpha_MH
        u_max_p_vals[i - 1] = u_max_p
        u = np.random.rand()                           
        if u < alpha_MH: # sample Unif(0, 1) to determine acceptance
            acc_indicator = 1.0                         
            samples_phi[i, :] = phi_p
            samples_theta[i, :] = theta_p
            sigma2_lhat_samples[i] = sigma2_lhat_p
            log_posthat_samples[i] = log_posthat_p 
            # Proposed becomes current in next iteration
            phi_c, theta_c, u_c, log_posthat_c, sigma2_lhat_c, log_q_c = phi_p, theta_p, u_p, log_posthat_p, sigma2_lhat_p, log_q_p
        else:
            acc_indicator = 0.0                         
            samples_phi[i, :] = phi_c 
            samples_theta[i, :] = theta_c 
            log_posthat_samples[i] = log_posthat_c 
            sigma2_lhat_samples[i] = sigma2_lhat_c

        if adapt:                                       
            a_i = 1.0 / i                              
            log_scale = log_scale + a_i * (acc_indicator - target_acc) 
                   
    run_time = time.time() - tic

    results = {"samples_theta": samples_theta, "samples_phi": samples_phi, "log_posthat_samples": log_posthat_samples,
               "sigma2_lhat_samples": sigma2_lhat_samples, "u_max_prop": u_max_p_vals, "alphas_MH": alphas_MH, 
               "outside_support": outside_support, "N": N, "run_time": run_time, "names": names, "dataset_name": dataset_name,
               "estimator_params": estimator_params}
    
    return results



## 2. Illustration using a simulated example
This section illustrates the code above through a simulated data example.

### 2.1 Specify simulation and model settings

In [ ]:
model_name = "ARMA11"  # WINSTON ADJUST
error_type = "normal"  # WINSTON ADJUST
T = 10000 #0 # Length of time series
p = 1 # Nbr of autoregressive y-lags in the GARCH-type variance process
q = 1 # Nbr of autoregressive sigma2-lags in the GARCH-type variance process
# Specify transforms (set all "identity" if no transformation)
dataset_name = "simulated_example_%s(%s,%s)_%s_error" % (model_name, p, q, error_type) # WINSTON ADJUST

param_names, trans_param_names = get_param_names(model_name, error_type, p, q)
transforms = ["bounded_(-1,1)", "bounded_(-1,1)", "log"] # WINSTON ADJUST

# Likelihood optimisation settings (posterior optimisation always uses L-BFGS-B as it has a better starting value):
likelihood_optim = "L-BFGS-B" #"basinhopping" # "L-BFGS-B" # "basinhopping" #"L-BFGS-B" #"basinhopping" #"L-BFGS-B" # "trust-ncg" # "L-BFGS-B" # "trust-ncg" # "L-BFGS-B" #"trust-ncg" #"L-BFGS-B" #"L-BFGS-B" 
options_L_BFGS_B = {"gtol":1e-6, "ftol":1e-12, "maxls":50, "maxiter":2000, "disp":True}
options_trust_ncg = {"gtol":1e-6, "maxiter":2000, "disp":True}
options_basinhopping = {"niter":50, "stepsize":0.5, "T":1.0, "disp":True}

# Prior settings:
# The default settings are
# mu_mean = 0.0, mu_sd = 10.0, omega_scale = 1.0, alpha_scale = 0.2, beta_scale = 0.8, v_shape = 2.0, v_rate = 1.0
# If other setting required, save as dictionary and pass on to the log prior once defined.
# Example:
prior_args = "default"
# prior_args = {"mu_mean":0.0, "mu_sd":100.0, "omega_scale":1.0, "alpha_scale":0.2, "beta_scale":0.8, "v_shape":2.0, "v_rate":1.0}

# Initialise prior. Either with default parameters or as specificed in prior_args
if prior_args == "default":
    log_prior = initiate_priors(model_name, error_type)
    prior_hyperparams = {} # WINSTON ADJUST
else:
    prior_hyperparams = prior_args
    log_prior = initiate_priors(model_name, error_type, **prior_hyperparams)

# Penalty settings:
penalty_args = "default"

# Inititialise penalties for frequentist estimation
if penalty_args == "default":
    log_penalty = initiate_penalties(model_name, error_type)
    penalty_hyperparams = {} # WINSTON ADJUST
else:
    penalty_hyperparams = penalty_args
    
        
# Create model functions based on the inputs above
h, hinv, hinv_p, hinv_pp, jacdiag, logdet = initiate_transformation_functions(transforms)
log_densities, grad_log_densities, Hess_log_densities = initiate_model_functions(model_name, h, hinv, hinv_p, hinv_pp) # WINSTON ADJUST
    
# Set true values (collected in theta0)    
theta0 = np_autograd.array([0.5, 0.2, 2])

phi0 = h(theta0) # True values in unrestricted space.
n_params = len(theta0)
assert(len(transforms) == n_params)

### 2.2 Set up folder and printing to PDF
This part of the code creates a folder where results will be stored. The folder name will be based on the name of the dataset. The code works by checking if a specific file (e.g. optimisation results) exists, and reads it in if that is the case. If it does not exist, then it is created (and will be read in the next time the code runs). The point of doing it this way is that we do not to do lengthy runs every time.

**WARNING**: Suppose that we simulate a dataset and subsequently run the code so that everything gets stored (control variates, optimisation, MCMC runs, etc). If the seed is then changed, so that the user generates another dataset (with the same name), then obviously the stored quantities are irrelevant. **It is the users responsibility to make sure that what is stored corresponds to the dataset used.** 

In [ ]:
# Keep text selectable in the PDF
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype']  = 42
# Latex
mpl.rcParams['text.usetex'] = False 

# Make a unique timestamped path
tz = ZoneInfo("Australia/Sydney")
stamp = datetime.now(tz).strftime("%Y%m%d-%H%M")  # e.g. 20251024-0712
project = "accelerating_ARMA11-run"                  # WINSTON ADJUST              
out_dir = Path("results_THESIS_ARMA11/%s/" % dataset_name)  # WINSTON ADJUST
out_dir.mkdir(parents=True, exist_ok=True)
PDF_PATH = out_dir / f"{project}_{stamp}.pdf"

# Create a single PdfPages object to reuse across cells

# Safely close any previous handle (ignore errors if it's already closed/not there)
with suppress(Exception):
    PDF.close()

# Open a fresh handle
PDF = PdfPages(PDF_PATH)

# Close safely at kernel exit (also ignores “already closed”)
def _pdf_close():
    with suppress(Exception):
        PDF.close()

atexit.register(_pdf_close)

print(f"Writing to: {PDF_PATH}")

with capture_output(display) as cap:
    print("==============================================================")
    print("model_name:", model_name, ", Error distribution:", error_type)
    print("Lags: p =, ", p, "q = ", q)
    print("Dataset name: ", dataset_name)
    print("Number of observations: ", T)
    print("Parameters:")
    display(Markdown(", ".join(param_names)))
    print("Tranformations:\n", transforms)
    print("Number of parameters:", n_params)
    # print("Prior hyperparameters (v_rate and v_shape only if Student-t)")
    # print_kv(prior_hyperparams)
    print("==============================================================")

text = cap.stdout
pdf_add_text(text, title="Settings specified")
cap.show()


### 2.3 Simulate data

In [ ]:
# Simulate data
y = simulate_ARMA11(T, theta0) # WINSTON ADJUST
     
# Measurements
plt.figure(figsize=(8, 3.5))
plt.title(f"Measurements ({error_type})")
plt.plot(y, color="cornflowerblue")
plt.xlabel(r"$t$")
plt.ylabel(r"$y_t$")
plt.tight_layout()

pdf_add_fig()


### 2.4 Check implementation (optional)
Check if the gradient and Hessian are correct by comparing to numerical differentiation.

In [ ]:
debug_gradient = False # False # NOTE: Make sure T is not too large if this is true. # WINSTON ADJUST
debug_Hessian = False # NOTE: Make sure T is not too large if this is true. # WINSTON ADJUST

if debug_gradient:
    assert T <= 10, "Set T smaller than 10 for debugging purposes."
    
    log_densities_for_numerical_gradient = lambda x: log_densities(hinv(x), y) # WINSTON ADJUST
        
    print("======================================================")
    print("Output to debug gradient (%s, %s):" % (model_name, error_type))
    print("======================================================")

    grad_l_t_all = grad_log_densities(theta0, y)  # WINSTON ADJUST
    print('Analytical gradient (first 2, last 2)')
    print(grad_l_t_all[:2, :])
    print(grad_l_t_all[-2:, :])

    
    # Debug the gradient by comparing to numerical differentiation.
    J_num = Jacobian(log_densities_for_numerical_gradient)(phi0)
    print('Numerical diff')
    print(J_num[:2, :])
    print(J_num[-2:, :])
    
    assert(np.allclose(grad_l_t_all, J_num))


if debug_Hessian:
    assert T <= 10, "Set T smaller than 10 for debugging purposes."
    
    # if model_name == "GARCH":
    #     Hess_num_func = lambda x: log_densities_GARCH_for_numerical_Hess(hinv(x), t, p, q, y[:t], y_start, sigma2_start, error_type) # diff with respect to phi
    # elif model_name == "TGARCH":
    #     Hess_num_func = lambda x: log_densities_TGARCH_for_numerical_Hess(hinv(x), t, p, q, y[:t], y_start, sigma2_start, error_type) # diff with respect to phi

    Hess_num_func = lambda x: log_densities_ARMA11_for_numerical_Hess(hinv(x), y[:t]) # WINSTON ADJUST

    print("======================================================")
    print("Output to debug Hessian (%s, %s):" % (model_name, error_type))
    print("======================================================")
    
    # Debug Hessian
    Hess_l_t_all = Hess_log_densities(theta0, y) # WINSTON ADJUST
    print(Hess_l_t_all.shape)
    T = len(y)
    for t in range(1, T + 1):
        print('------------------------------------------------------')
        print('t = ', t)
        print('Analytical \n', Hess_l_t_all[t-1, :, :])
        # Hess_num_func = lambda x: log_densities_ARMA11(hinv(x), y)[t-1]   # WINSTON ADJUST
        Hess_num = Hessian(Hess_num_func)(phi0)
        print('Numerical diff \n', Hess_num)
        check_Hessian_close(Hess_l_t_all[t-1, :, :], Hess_num)

### 2.6 Time evaluations 
This subsection times the densities, gradients, and Hessians runs.

In [ ]:
with capture_output() as cap:
    print("==============================================================================================================")
    start = time.perf_counter()
    result = log_densities(theta0, y, model_name) # WINSTON ADJUST
    end = time.perf_counter()
    print("Elapsed time log-densities evaluation with T = " , T, "p = " , p, "q = " , q, ":", end - start, "seconds")

    start = time.perf_counter()
    result = grad_log_densities(theta0, y, model_name) # WINSTON ADJUST
    end = time.perf_counter()
    print("Elapsed time grad log-densities evaluation with T = " , T, "p = " , p, "q = " , q,  ":", end - start, "seconds")

    start = time.perf_counter()
    result = Hess_log_densities(theta0, y, model_name) # WINSTON ADJUST
    end = time.perf_counter()
    print("Elapsed time Hess log-densities evaluation with T = " , T, "p = " , p, "q = " , q,  ":", end - start, "seconds")
    print("==============================================================================================================")

text = cap.stdout
pdf_add_text(text, title="Time evaluations")
cap.show()

### 2.7 Maximum likelihood optimisation using the full data
We now perform optimisation to find the maximum likelihood estimate (MLE) to check if we can recover true parameters. The optimisation is performed on the unrestricted parameter $\boldsymbol{\phi}$.

The following code maximises the log-likelihood and constructs asymptotic confidence intervals (for $\boldsymbol{\phi}$). Note that although the optimisation function (obj_func_likelihood) for the log-likelihood is unrestricted, it does not inforce stationarity.

In [ ]:
# Minimise in phi space, so input is unrestricted
def log_penalty_phi(x):
    return log_penalty(hinv(x)) # WINSTON ADJUST

def obj_func_likelihood(x):
    """
    Penalised log-likelihood. To enforce the
    """
    return -log_likelihood(hinv(x), y, model_name)/T + log_penalty_phi(x)/T # WINSTON ADJUST

grad_log_penalty = grad(log_penalty_phi) #(x, p, q)
Hess_log_penalty = hessian(log_penalty_phi)

# WINSTON ADJUST
grad_obj_func_likelihood = lambda x: -grad_log_likelihood(hinv(x), y, model_name)/T + grad_log_penalty(x)/T # Gradient of obj_func_likelihood. Computed by automatic differentiation
Hess_obj_func_likelihood = lambda x: -Hess_log_likelihood(hinv(x), y, model_name)/T + Hess_log_penalty(x)/T # Hessian of obj_func_likelihood. Computed by automatic differentiation

optim_dir = out_dir / "optim"
optim_dir.mkdir(parents=True, exist_ok=True)
save_path_optim = optim_dir / "optim_likelihood_result.h5"

# Optimise the log-likelihood
start = time.perf_counter()
it = {"k": 0} # For callback
fg = FGCache(obj_func_likelihood, grad_obj_func_likelihood)

try:
    optim_results = load_dict_h5(save_path_optim)
    MLE = np.array(optim_results['optim_obj']['x'])
    Cov = optim_results['Cov_MLE']
    print('95% Confidence intervals for MLE estimates (phi)')
    print(np.round(list(MLE - 1.96*np.sqrt(np.diag(Cov))), 2))
    print(np.round(list(MLE + 1.96*np.sqrt(np.diag(Cov))), 2))
    print("\n\n")

except FileNotFoundError:
    theta_optim_start = np_autograd.array([0.5, 0.2, 2])

    phi_optim_start = h(theta_optim_start)
    
    start = time.perf_counter()
    it = {"k": 0} # For callback
    fg = FGCache(obj_func_likelihood, grad_obj_func_likelihood)

    start = time.perf_counter()
    it = {"k": 0} # For callback
    fg = FGCache(obj_func_likelihood, grad_obj_func_likelihood)

    if likelihood_optim == "trust-ncg":
        print('\n\nLikelihood optimisation settings for method trust-ncg:\n')
        options = options_trust_ncg
        print_kv(options)
        res_optim_likelihood = minimize(fg.fun, phi_optim_start, jac=fg.jac, hess=Hess_obj_func_likelihood, 
                                    method="trust-exact", options=options)  

    elif likelihood_optim == "L-BFGS-B":
        print('\n\nLikelihood optimisation settings for method L-BFGS-B:\n')
        options = options_L_BFGS_B
        print_kv(options)
        res_optim_likelihood = minimize(fg.fun, phi_optim_start, method='L-BFGS-B', jac= fg.jac, callback = callback_optim,
                                        options=options)
    elif likelihood_optim == "basinhopping":
        minimizer_kwargs = {"method": "L-BFGS-B", "jac": fg.jac, "options": options_L_BFGS_B}
        res_optim_likelihood = basinhopping(fg.fun, x0 = phi_optim_start, minimizer_kwargs=minimizer_kwargs,
                                            callback = callback_optim, **options_basinhopping)
    else:
        raise ValueError("Optimisation method does not exist")        

    end = time.perf_counter()
    run_time = end - start

    with capture_output() as cap:
        print("==============================================================")
        print("\nLikelihood optimisation took", end - start, "seconds.")        

        print('Starting values: \n', np.round(phi_optim_start, 3))
        # Compare answers
        print('True parameter values (phi)')
        print(np.round(phi0, 2))
        print('MLE estimates (phi)')
        MLE = res_optim_likelihood.x
        print(np.round(MLE, 2))
        print('True parameter values (theta)')
        print(theta0)
        print('hinv(MLE) estimates (theta)')
        print(np.round(hinv(MLE), 2))

        print('95% Confidence intervals for MLE estimates (phi)')
        Cov = np.linalg.inv(Hess_obj_func_likelihood(MLE))/T # Negative Hessian inverse evaluated at the mode
        print(np.round(list(MLE - 1.96*np.sqrt(np.diag(Cov))), 2))
        print(np.round(list(MLE + 1.96*np.sqrt(np.diag(Cov))), 2))
        print("\n\n")
        
        if likelihood_optim == "basinhopping":
            print_minimize_summary(res_optim_likelihood["lowest_optimization_result"], x_names=trans_param_names, latex=True)
        else:
            print_minimize_summary(res_optim_likelihood, x_names=trans_param_names, latex=True)

        print("==============================================================")
    text = cap.stdout
    pdf_add_text(text, title="Optimisation of log-likelihood. Summary.")
    cap.show()
    
    optim_results = {'optim_obj': pack_optimize_result(res_optim_likelihood), 'method': likelihood_optim, 'Cov_MLE': Cov, 'start_value': phi_optim_start}
    save_dict_h5(save_path_optim, optim_results)


### 2.8 Maximum aposteriori optimisation using the full data
We now perform optimisation to find the maximum aposterior (MAP) estimate. This is needed to construct our control variates, and is also useful for the random-walk MH proposal later. The optimisation is performed on the unrestricted parameter $\boldsymbol{\phi}$. Credible intervals for the (phi) parameters are provided, based on a normal approximation of the (transformed) posterior.

This step uses the MLE as starting value, so it is significantly faster than the optimisation that obtains the MLE (starting much closer to the mode). For this reason, we do not save this step to a hd5 file.

**NOTE**: 
The subsampling MCMC is only stable for cases when the negative Hessian at the posterior mode is positive definite. This need not hold for GARCH($p$,$q$) models, particularly when $q > 2$, due to weak identification among the volatility parameters. In such cases, we estimate the Fisher information by the sum of outer product of gradient observations. While this often results in a matrix that is positive definite, it may overestimate the posterior variation. This may cause issues since the proposal may be far from where the posterior mass is located, resulting in a potentially inaccurate control variate. We have employed samplers that, when the Fisher is estimated this way, start with a small scaling factor and adapt it to have a target acceptance probability of 0.23. While this may help subsampling MCMC, it is not guaranteed to work. Take away message: subsampling MCMC works reliaibly for cases where the observed Fisher information is invertible. 

In [ ]:
# Define gradients and Hessian for log-posterior
log_prior_phi = lambda x: log_prior(hinv(x)) + logdet(x) # for autograd  # WINSTON ADJUST
grad_log_prior_phi = grad(log_prior_phi)
grad_log_posterior = lambda x: grad_log_likelihood(hinv(x), y, model_name) + grad_log_prior_phi(x) # WINSTON ADJUST
Hess_log_prior_phi = hessian(log_prior_phi)
Hess_log_posterior = lambda x: Hess_log_likelihood(hinv(x), y, model_name) + Hess_log_prior_phi(x) # WINSTON ADJUST

obj_func_posterior = lambda x: -log_posterior(x, y, model_name)[0] # Input is phi  # WINSTON ADJUST
grad_obj_func_posterior = lambda x: -grad_log_posterior(x) 
Hess_obj_func_posterior = lambda x: -Hess_log_posterior(x)

# Optimise the log-posterior in phi space, so input is unrestricted
phi_optim_start = MLE
start = time.perf_counter()
it = {"k": 0} # For callback
fg = FGCache(obj_func_posterior, grad_obj_func_posterior)

res_optim_posterior = minimize(fg.fun, phi_optim_start, method='L-BFGS-B', jac= fg.jac, callback = callback_optim,
                                options=options_L_BFGS_B)
end = time.perf_counter()
with capture_output() as cap:
    print("==============================================================")

    print("\nPosterior optimisation took", end - start, "seconds.")
    print('\n\nPosterior optimisation settings for method L-BFGS-B:\n')
    print_kv(options_L_BFGS_B)
    
    print('Starting values: \n', np.round(phi_optim_start, 3))
    # Compare answers
    print('True parameter values (phi)')
    print(np.round(phi0, 2))
    print('MAP estimates (phi)')
    MAP = res_optim_posterior.x
    print(np.round(MAP, 2))
    print('True parameter values (theta)')
    print(theta0)
    print('hinv(MAP) estimates (theta)')
    print(np.round(hinv(MAP), 2))

    print('95% Credible intervals (normality assumption on posterior) (phi)')
    Hess_MAP = Hess_obj_func_posterior(MAP)
    Cov = np.linalg.inv(Hess_MAP) # Negative Hessian inverse evaluated at the mode  
    if (np.linalg.eigvalsh(Cov) < 0).any():
        print("Cannot compute posterior covariance from Hessian. Trying outer product approximation")
        scores = grad_log_densities(hinv(MAP), y) # WINSTON ADJUST
        Cov = np.linalg.inv(scores.T @ scores)
        assert (np.linalg.eigvalsh(Cov) > 0).all(), "Cannot compute posterior covariance."
        adapt_scaling = True
    else:
        adapt_scaling = False
    print(np.round(list(MAP - 1.96*np.sqrt(np.diag(Cov))), 2))
    print(np.round(list(MAP + 1.96*np.sqrt(np.diag(Cov))), 2))
    print("\n\n")
    print_minimize_summary(res_optim_posterior, x_names=trans_param_names, latex=True)
    
    print("==============================================================")

    
text = cap.stdout
pdf_add_text(text, title="Optimisation of log-posterior. Display and summary.")
cap.show()


### 2.9 Sampling the full data posterior (NOTE: Maybe computationally expensive for large $T$)
This section uses a Metropolis-Hastings sampler to sample from the the full data posterior distribution. This will serve as the ground truth to compare our subsampling MCMC and variational Bayes intractable log-likelihood results against. It will also help us establish the support of the posterior to evaluate the quality of the control variates.

In [ ]:
# MCMC settings. Samples and prop_cov
N = 5500 # MCMC samples
MCMC_dir = out_dir / "MCMC"
MCMC_dir.mkdir(parents=True, exist_ok=True)
save_path_MCMC = MCMC_dir / ("full_data_MCMC_N%s.h5" % N)

try:
    MCMC_results = load_dict_h5(save_path_MCMC)
    result_full_data_MCMC = MCMC_results['MCMC_obj']
    prop_cov = MCMC_results['prop_cov']
    # Convert back to arrays
    for key in ["samples_phi", "samples_theta"]:
        result_full_data_MCMC[key] = np.asarray(result_full_data_MCMC[key])

except FileNotFoundError:
    # Run MCMC
    Sigma_pi = Cov # Negative Hessian inverse evaluated at the mode
    prop_cov = 2.38**2/n_params*Sigma_pi # For the random walk Negative Hessian inverse evaluated at the mode

    phi_init = MAP # + sps.norm.rvs(0, 0.1*np.sqrt(np.diag(Sigma_pi)), size = len(MAP))
    names = {"params_name": param_names, "trans_param_names": trans_param_names}
    print("==============================================================")
    result_full_data_MCMC = sample_full_data_posterior_random_walk(phi_init, prop_cov, N, names, dataset_name, adapt = adapt_scaling)
    print("==============================================================")

    MCMC_results = {'MCMC_obj': result_full_data_MCMC, 'prop_cov': prop_cov, 'start_value': phi_init}
    save_dict_h5(save_path_MCMC, MCMC_results)


In [ ]:
# Plots
plot_credible_intervals(result_full_data_MCMC['samples_phi'], param_true=phi0, param_names=trans_param_names, use_median=False)
plot_credible_intervals(result_full_data_MCMC['samples_theta'], param_true=theta0, param_names=param_names, use_median=False)
plot_traceplots_MCMC(result_full_data_MCMC['samples_phi'], 
                param_true = phi0, param_names=trans_param_names,
                burnin=0, thin=1, use_median=False, show_running_mean=True, runmean_window=50, 
                suptitle="Traceplots full data MCMC")
plot_traceplots_MCMC(result_full_data_MCMC['samples_theta'], 
                param_true = theta0, param_names=param_names,
                burnin=0, thin=1, use_median=False, show_running_mean=True, runmean_window=50, 
                suptitle="Traceplots full data MCMC")
pdf_add_fig()

### 2.10 Initialise and evaluate control variates

In [ ]:
phiStar = MAP
args = (y, model_name)  # WINSTON ADJUST
control_variate_dir = out_dir / "control_variates"
control_variate_dir.mkdir(parents=True, exist_ok=True)
save_path_control_variates = control_variate_dir / "control_variate_result.h5"
try:
    control_variates = load_dict_h5(save_path_control_variates)
    dens_at_phiStar = control_variates['dens_at_phiStar']
    grad_at_phiStar = control_variates['grad_at_phiStar'] 
    Hess_at_phiStar = control_variates['Hess_at_phiStar']
    end = control_variates['end']
    start = control_variates['start']

except FileNotFoundError:
    start = time.perf_counter()
    dens_at_phiStar,  grad_at_phiStar, Hess_at_phiStar = initiate_control_variate_quantities(log_densities, grad_log_densities, Hess_log_densities, phiStar, args)
    end = time.perf_counter()
    text = cap.stdout
    control_variates = {'dens_at_phiStar': dens_at_phiStar, 'grad_at_phiStar': grad_at_phiStar, 
                        'Hess_at_phiStar': Hess_at_phiStar, 'end': end, 'start': start}
    save_dict_h5(save_path_control_variates, control_variates)


with capture_output() as cap:
    print("==============================================================")
    print("Elapsed time control variates initialisation with T = " , T, "p = " , p, "q = " , q,  ":", end - start, "seconds")
    print("==============================================================")
text = cap.stdout
pdf_add_text(text, title="Control variate construction.")
cap.show()

# Precompute quantities for the control variates that do not depend on theta
A = np.sum(dens_at_phiStar) 
B = np.sum(grad_at_phiStar, axis = 0)
C = np.sum(Hess_at_phiStar, axis = 0)
# sum of control variates evaluated at phi    
q_sum = lambda phi: A + np.dot(B, phi - phiStar) + 0.5*np.dot(phi - phiStar, np.dot(C, phi - phiStar))


The accuracy of the control variates is evaluated for different orders (0, 1, 2) of the Taylor approximation. The control variates are evaluated using samples $\boldsymbol{\phi}$ from the normal approximation of the posterior distribution. The samples are constructed to be at increasingly further distances from the posterior mode. 

In [ ]:
# Outside median ellipse in phi space
Sigma_pi = Cov # np.linalg.inv(Hess_obj_func_posterior(MAP))
phi = mvn_tail_sample(MAP, Sigma_pi, q = 0.50, n = 1).flatten()
phi_beyond_median = phi
q_k_order0 = eval_q_k(phi, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 0) 
q_k_order1 = eval_q_k(phi, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 1) 
q_k_order2 = eval_q_k(phi, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 2) 
l_k = log_densities(hinv(phi), *args)

fig, axs = plt.subplots(1, 3, constrained_layout=True, figsize=(8.5, 3.2))
fig.suptitle(
    r'%s(%s, %s) %s. $\phi$ outside median ellipse of $\phi$' % (model_name, p, q, error_type) + r'(, $||\phi - \phi^\star|| = %3.4f)$' % np.linalg.norm(phi - phiStar),
    fontsize=14, y=0.98   # try 0.99 if you want it even closer
)
fig.set_constrained_layout_pads(h_pad=0.4, w_pad=0.05, hspace=0.08, wspace=0.08)
for k in range(3):
        axs[k].plot(l_k, l_k, color = 'red')
        if k == 0:
            axs[k].plot(l_k, q_k_order0, '.', color = 'cornflowerblue')
            axs[k].set_title('Taylor order 0', size = 12)
        elif k == 1:
            axs[k].plot(l_k, q_k_order1, '.', color = 'cornflowerblue')
            axs[k].set_title('Taylor order 1', size = 12)
        elif k == 2:
            axs[k].plot(l_k, q_k_order2, '.', color = 'cornflowerblue')
            axs[k].set_title('Taylor order 2', size = 12)
            
        axs[k].axis('scaled')


# Outside 0.999 ellipse
phi = mvn_tail_sample(MAP, Sigma_pi, q = 0.999, n = 1).flatten()
phi_extreme = phi
q_k_order0 = eval_q_k(phi, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 0) 
q_k_order1 = eval_q_k(phi, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 1) 
q_k_order2 = eval_q_k(phi, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 2) 
l_k = log_densities(hinv(phi), *args)

fig, axs = plt.subplots(1, 3, constrained_layout=True, figsize=(8.5, 3.2))
fig.suptitle(
    r'%s(%s, %s) %s. $\phi$ outside 0.999 ellipse of $\phi$' % (model_name, p, q, error_type) + r'(, $||\phi - \phi^\star|| = %3.4f)$' % np.linalg.norm(phi - phiStar),
    fontsize=14, y=0.98   # try 0.99 if you want it even closer
)
fig.set_constrained_layout_pads(h_pad=0.4, w_pad=0.05, hspace=0.08, wspace=0.08)
for k in range(3):
        axs[k].plot(l_k, l_k, color = 'red')
        if k == 0:
            axs[k].plot(l_k, q_k_order0, '.', color = 'cornflowerblue')
            axs[k].set_title('Taylor order 0', size = 12)
        elif k == 1:
            axs[k].plot(l_k, q_k_order1, '.', color = 'cornflowerblue')
            axs[k].set_title('Taylor order 1', size = 12)
        elif k == 2:
            axs[k].plot(l_k, q_k_order2, '.', color = 'cornflowerblue')
            axs[k].set_title('Taylor order 2', size = 12)
            
        axs[k].axis('scaled')


### 2.11 Truncated power-law decaying sampling probabilities
Compute and plot the probability mass function for two cases. $\gamma = 0$ (uniform weights) and $\gamma > 0$.

In [ ]:
# Check probability mass function
# Case 1: gamma = 0. Should provide uniform weights
t_star = int(0.01*T)
b = 100.0

# Compute pmf
gamma_val = 0.0
probs, eps = TPD_probs(T,t_star, gamma = gamma_val, b = b)

# Plot the pmf
plt.figure(figsize=(9,4))
plt.plot(range(1, T+1), probs, marker='o', markersize=2)
plt.axvline(t_star+0.5, color='red', linestyle='--', label='t*')
plt.xlabel("t")
plt.ylabel(r"$p_t$")
plt.title(r"Case 1: Probability mass function TPD when $\gamma = %3.3f$" % gamma_val)
plt.legend()

# Print diagnostics
print("==============================================================================================================")
print("Case 1:")
print("T = ,", T, "tstar = ", t_star)
print("Case gamma = ", gamma_val)
print("Head mass (empirical):", probs[:t_star].sum())
print("Tail mass (empirical):", probs[t_star:].sum())
print("Head min prob:", probs[:t_star].min())
print("Tail min prob:", probs[t_star:].min())
print("Reference: Uniform min:", 1/T)
print("Tail ratio (baseline uniform)", probs[t_star:][0]/(1/T))
print("Reference: eps from function:", eps)
print("Which c: ", T/(T - t_star)*eps)
print("Sum of probs:", probs.sum())

# Case 2
# Compute pmf
gamma_val = 14.0
probs, eps = TPD_probs(T, t_star, gamma = gamma_val, b = b)

# Plot the pmf
plt.figure(figsize=(9,4))
plt.plot(range(1, T+1), probs, marker='o', markersize=2)
plt.axvline(t_star+0.5, color='red', linestyle='--', label='t*')
plt.xlabel("t")
plt.ylabel(r"$p_t$")
plt.title("Case 2: Probability mass function TPD when $\gamma = %3.3f$" % gamma_val)
plt.legend()

# Print diagnostics
print("\n")
print("Case 2:")
print("Case gamma = ", gamma_val)
print("Head mass (empirical):", probs[:t_star].sum())
print("Tail mass (empirical):", probs[t_star:].sum())
print("Head min prob:", probs[:t_star].min())
print("Tail prob:", probs[t_star:][0])
print("Reference: Uniform min:", 1/T)
print("Tail ratio (baseline uniform)", probs[t_star:][0]/(1/T))
print("Reference: eps from function:", eps)
print("Which c: ", T/(T - t_star)*eps)
print("Sum of probs:", probs.sum())
print("==============================================================================================================")

### 2.12 Compute variance and expected computational cost
This section shows how the variance and expected computational cost are computed

In [ ]:
# # MAP and Perturbed MAP
phi_MAP = MAP
phi_MAP_perturbed = MAP + sps.norm.rvs(scale = 0.01*np.diag(Sigma_pi), size = n_params)


### 2.13 Subsampling MCMC
This section shows how to run subsampling MCMC. We start by tuning the algorithm, i.e. choosing with $m$ and $c$ to use for constructing the sampling weights. Note that setting a $c$ implies the largest $\gamma_{\mathrm{max}}$ compatible with the floor $\varepsilon(\gamma) \geq c(T-t^\star)/T$; see the paper for details. 

We consider two distinct approaches for the tuning:

1. Minimising the variance of the estimator under a computational constraint.
2. Minimising the computational cost of the estimator under a variance constraint.

Both approaches require the population $e_t =  \ell_t - q_t$, which requires knowledge of $\boldsymbol{\theta}$. We tune the algorithm based on a typical $\boldsymbol{\theta}$, such as one close to the posterior maximum aposteriori (MAP) estimate. For the first approach, we consider an additional tuning strategy that does not depend on the $e_t$, because it minimises an upper bound of the variance. See the paper for details.

The natural approach for subsampling MCMC is to minimise compute given a variance tolerance budget (approach 2), as the literature provides guidelines on what an optimal variance (around 1) should be.

In [ ]:
# First tune the algorithm. Tune around theta_MAP_perturbed
phi_tune_approach2 = phi_MAP_perturbed
q_k_order2 = eval_q_k(phi_tune_approach2, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 2)
e_t_tune_approach2 = log_densities(hinv(phi_tune_approach2), *args) - q_k_order2 # Population elements to tune at
V_tolerance = 1 # WINSTON ADJUST
c_opt, m_opt = tune_c_m_approach2(V_tolerance, e_t_tune_approach2)

gamma_opt = gamma_max_value(T, t_star, c_opt, b)
probs, eps = TPD_probs(T, t_star, gamma = gamma_opt, b = b)
with capture_output() as cap:
    print("==============================================================")
    print("Tuning approach 2 (minimise expected compute given variance tolerance)")
    print("c_opt =", c_opt, "m_opt =", m_opt)
    print("Implied gamma_max =", np.round(gamma_opt))    
    print("==============================================================")
    
text = cap.stdout
pdf_add_text(text, title="Tuning of subsampling MCMC.")
cap.show()

# Plot optimal probabilities
# Compute pmf
probs, eps = TPD_probs(T, t_star, gamma = gamma_opt, b = b)
# Plot the pmf
plt.figure(figsize=(9,4))
plt.plot(range(1, T+1), probs, marker='o', markersize=2)
plt.axvline(t_star+0.5, color='red', linestyle='--', label='t*')
plt.xlabel("t")
plt.ylabel(r"$p_t$")
plt.title(r"Approach 2 tune when $V = %3.3f$: Pmf TPD when $\gamma = %3.3f$" % (V_tolerance, gamma_opt))
plt.legend()
pdf_add_fig()



In [ ]:
N = 32000
save_path_subsampling_MCMC = MCMC_dir / ("subsampling_MCMC_N%s.h5" % N)

try:
    subsampling_MCMC_results = load_dict_h5(save_path_subsampling_MCMC)
    result_subsampling_MCMC = subsampling_MCMC_results['MCMC_obj']
    prop_cov = subsampling_MCMC_results['prop_cov']
    # Convert back to arrays
    for key in ["samples_phi", "samples_theta"]:
        result_subsampling_MCMC[key] = np.asarray(result_subsampling_MCMC[key])

except FileNotFoundError:
    # Run Subsampling MCMC
    estimator_params = {"m_opt": m_opt, "c_opt": c_opt, "gamma_opt": gamma_opt, "t_star": t_star, "b":b, 
                        "tuning_approach": "Approach 2", "tuning_budget": V_tolerance}
    phi_init = MAP 
    prop_cov = 2.38**2/n_params*Sigma_pi # For the random walk Negative Hessian inverse evaluated at the mode
    names = {"params_name": param_names, "trans_param_names": trans_param_names}
    print("==============================================================")
    result_subsampling_MCMC = sample_subsampling_posterior_random_walk(estimator_params, phi_init, prop_cov, N, names, dataset_name, adapt = adapt_scaling)
    print("==============================================================")
    subsampling_MCMC_results = {'MCMC_obj': result_subsampling_MCMC, 'estimator_params': estimator_params, 'prop_cov': prop_cov, 'start_value': phi_init}
    save_dict_h5(save_path_subsampling_MCMC, subsampling_MCMC_results)




In [ ]:
# Plot results
burn_in = 500
plot_credible_intervals(result_subsampling_MCMC['samples_phi'], param_true=phi0, param_names=trans_param_names, use_median=False)
plot_credible_intervals(result_subsampling_MCMC['samples_theta'], param_true=theta0, param_names=param_names, use_median=False)
plot_traceplots_MCMC(result_subsampling_MCMC['samples_phi'], 
                param_true = phi0, param_names=trans_param_names,
                burnin=0, thin=1, use_median=False, show_running_mean=True, runmean_window=50, 
                suptitle = "Traceplots subsampling MCMC")
plot_subsampling_MCMC_quantities(result_subsampling_MCMC)

tau_full = np.array([IACT(result_full_data_MCMC['samples_phi'][burn_in:, item]) for item in range(n_params)])
tau_sub = np.array([IACT(result_subsampling_MCMC['samples_phi'][burn_in:, item]) for item in range(n_params)])
plot_inefficiency_comparison(tau_full, tau_sub, param_names = trans_param_names)

tau_full = np.array([IACT(result_full_data_MCMC['samples_theta'][burn_in:, item]) for item in range(n_params)])
tau_sub = np.array([IACT(result_subsampling_MCMC['samples_theta'][burn_in:, item]) for item in range(n_params)])
plot_inefficiency_comparison(tau_full, tau_sub, param_names = param_names)


plot_kde_comparison(result_full_data_MCMC['samples_phi'][burn_in:, :], result_subsampling_MCMC['samples_phi'][burn_in:, :], 
                    param_names = trans_param_names, title="Posterior marginals in phi-space: Full vs Subsampling MCMC", 
                    n_points=200, true_vals=phi0, labels=("Full MCMC", "Subsampling"))
plot_kde_comparison(result_full_data_MCMC['samples_theta'][burn_in:, :], result_subsampling_MCMC['samples_theta'][burn_in:, :], 
                    param_names = param_names, title="Posterior marginals in theta-space: Full vs Subsampling MCMC", 
                    n_points=200, true_vals=theta0, labels=("Full MCMC", "Subsampling"))

pdf_add_fig()
_pdf_close() # Close the pdf that contains the generated figures and text.  # WINSTON ADJUST


## 3. Empircal application: Discrete-time Vasicek model for SOFR
This section contains code for Section 4 in the thesis

### 3.1 Specify simulation and model settings

In [ ]:
model_name = "Vasicek_Demeaned"  # WINSTON ADJUST
error_type = "normal"  # WINSTON ADJUST
p = 1 # Nbr of autoregressive y-lags in the GARCH-type variance process
q = 0 # Nbr of autoregressive sigma2-lags in the GARCH-type variance process
# Specify transforms (set all "identity" if no transformation)
dataset_name = "simulated_example_%s(%s,%s)_%s_error" % (model_name, p, q, error_type) # WINSTON ADJUST

param_names, trans_param_names = get_param_names(model_name, error_type, p, q)
if model_name == "Vasicek":
    transforms = ["bounded_(-1,1)", "log", "log", "log"] # WINSTON ADJUST
elif model_name == "Vasicek_Demeaned":
    transforms = ["bounded_(-1,1)", "log", "log"] # WINSTON ADJUST
else:
    raise ValueError("Invalid model name")

# Likelihood optimisation settings (posterior optimisation always uses L-BFGS-B as it has a better starting value):
likelihood_optim = "L-BFGS-B" #"basinhopping" # "L-BFGS-B" # "basinhopping" #"L-BFGS-B" #"basinhopping" #"L-BFGS-B" # "trust-ncg" # "L-BFGS-B" # "trust-ncg" # "L-BFGS-B" #"trust-ncg" #"L-BFGS-B" #"L-BFGS-B" 
options_L_BFGS_B = {"gtol":1e-6, "ftol":1e-12, "maxls":50, "maxiter":2000, "disp":True}
options_trust_ncg = {"gtol":1e-6, "maxiter":2000, "disp":True}
options_basinhopping = {"niter":50, "stepsize":0.5, "T":1.0, "disp":True}

# Prior settings:
# The default settings are
# mu_mean = 0.0, mu_sd = 10.0, omega_scale = 1.0, alpha_scale = 0.2, beta_scale = 0.8, v_shape = 2.0, v_rate = 1.0
# If other setting required, save as dictionary and pass on to the log prior once defined.
# Example:
prior_args = "default"
# prior_args = {"mu_mean":0.0, "mu_sd":100.0, "omega_scale":1.0, "alpha_scale":0.2, "beta_scale":0.8, "v_shape":2.0, "v_rate":1.0}

# Initialise prior. Either with default parameters or as specificed in prior_args
if prior_args == "default":
    log_prior = initiate_priors(model_name, error_type)
    prior_hyperparams = {} # WINSTON ADJUST
else:
    prior_hyperparams = prior_args
    log_prior = initiate_priors(model_name, error_type, **prior_hyperparams)

# Penalty settings:
penalty_args = "default"

# Inititialise penalties for frequentist estimation
if penalty_args == "default":
    log_penalty = initiate_penalties(model_name, error_type)
    penalty_hyperparams = {} # WINSTON ADJUST
else:
    penalty_hyperparams = penalty_args
    
        
# Create model functions based on the inputs above
h, hinv, hinv_p, hinv_pp, jacdiag, logdet = initiate_transformation_functions(transforms)
log_densities, grad_log_densities, Hess_log_densities = initiate_model_functions(model_name, h, hinv, hinv_p, hinv_pp) # WINSTON ADJUST
    
# Set true values (collected in theta0)    
theta0 = np_autograd.array([0.4, 0.002, 0.3])

phi0 = h(theta0) # True values in unrestricted space.
n_params = len(theta0)
assert(len(transforms) == n_params)

### 3.2 Set up folder and printing to PDF
This part of the code creates a folder where results will be stored. The folder name will be based on the name of the dataset. The code works by checking if a specific file (e.g. optimisation results) exists, and reads it in if that is the case. If it does not exist, then it is created (and will be read in the next time the code runs). The point of doing it this way is that we do not to do lengthy runs every time.

**WARNING**: Suppose that we simulate a dataset and subsequently run the code so that everything gets stored (control variates, optimisation, MCMC runs, etc). If the seed is then changed, so that the user generates another dataset (with the same name), then obviously the stored quantities are irrelevant. **It is the users responsibility to make sure that what is stored corresponds to the dataset used.** 

In [ ]:
# Keep text selectable in the PDF
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype']  = 42
# Latex
mpl.rcParams['text.usetex'] = False 

# Make a unique timestamped path
tz = ZoneInfo("Australia/Sydney")
stamp = datetime.now(tz).strftime("%Y%m%d-%H%M")  # e.g. 20251024-0712
project = "accelerating_Vasicek-run"                  # WINSTON ADJUST              
out_dir = Path("results_THESIS_Vasicek/%s/" % dataset_name)  # WINSTON ADJUST
out_dir.mkdir(parents=True, exist_ok=True)
PDF_PATH = out_dir / f"{project}_{stamp}.pdf"

# Create a single PdfPages object to reuse across cells

# Safely close any previous handle (ignore errors if it's already closed/not there)
with suppress(Exception):
    PDF.close()

# Open a fresh handle
PDF = PdfPages(PDF_PATH)

# Close safely at kernel exit (also ignores “already closed”)
def _pdf_close():
    with suppress(Exception):
        PDF.close()

atexit.register(_pdf_close)

print(f"Writing to: {PDF_PATH}")

with capture_output(display) as cap:
    print("==============================================================")
    print("model_name:", model_name, ", Error distribution:", error_type)
    print("Lags: p =, ", p, "q = ", q)
    print("Dataset name: ", dataset_name)
    print("Number of observations: ", T)
    print("Parameters:")
    display(Markdown(", ".join(param_names)))
    print("Tranformations:\n", transforms)
    print("Number of parameters:", n_params)
    print("==============================================================")

text = cap.stdout
pdf_add_text(text, title="Settings specified")
cap.show()


### 3.3 Query data

In [ ]:
# start = "1985-01-01"
# series = "T10Y2Y"   # term spread 10Y-2Y

# spread = web.DataReader(series, "fred", start)
# y_raw = spread[series].dropna().to_numpy() / 100.0  # đổi sang đơn vị 1
# y = y_raw - y_raw.mean()                            # center cho dễ
# T = len(y)

In [ ]:
# def fred(series_id: str) -> pd.Series:
#     url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
#     df = pd.read_csv(url)
#     df["DATE"] = pd.to_datetime(df["DATE"])
#     s = (df.set_index("DATE")[series_id]
#            .replace(".", pd.NA)
#            .dropna()
#            .astype(float))
#     return s

# sofr = fred("SOFR")   # Secured Overnight Financing Rate
# effr = fred("EFFR")   # Effective Federal Funds Rate

# df = pd.concat([sofr.rename("SOFR"), effr.rename("EFFR")], axis=1).dropna()
# df["SOFR_minus_EFFR"] = df["SOFR"] - df["EFFR"]      # percentage points
# df["SOFR_minus_EFFR_bp"] = df["SOFR_minus_EFFR"] * 100  # basis points

# spread = df["SOFR_minus_EFFR"]
# spread_bp = df["SOFR_minus_EFFR_bp"]

In [ ]:
start = "2016-01-01"
sofr = web.DataReader("SOFR", "fred", start)   # Effective Federal Funds Rate
sofr = sofr.dropna()/100
# effr

In [ ]:
import pandas_datareader.data as web
# y10 = web.DataReader("WGS10YR", "fred", start)["WGS10YR"].dropna() / 100.0
# y10 = web.DataReader("IRLTLT01USM156N", "fred", start)["IRLTLT01USM156N"].dropna() / 100.0 good for subsampling # weekly
# y10 = web.DataReader("WGS10YR", "fred", start)["WGS10YR"].dropna() / 100.0 # weekly
# y10 = web.DataReader("GS10", "fred", start)["GS10"].dropna() / 100.0 # monthly # good for full data # nhieu data monthly nhất
# y10 = web.DataReader("IRLTLT01USM156N", "fred", start)["IRLTLT01USM156N"].dropna() / 100.0 # weekly # good for subsampling
# y10 = web.DataReader("GS2", "fred", start)["GS2"].dropna() / 100.0 # monthly # good for full data # CURRENTLY BEST
# y10 = web.DataReader("GS5", "fred", start)["GS5"].dropna() / 100.0 # monthly # good for full data
# y10 = web.DataReader("WGS10YR", "fred", start)["WGS10YR"].dropna() / 100.0 
# y10 = web.DataReader("IRLTLT01AUM156N", "fred", start)["IRLTLT01AUM156N"].dropna() / 100.0 




In [ ]:
y = sofr.diff().dropna().to_numpy()
# y = y10.diff().dropna().to_numpy()
# y = y[-2000:]
# y = y[-3000:-1500]
T = len(y)
T

plt.plot(y)
plt.ylim(-0.005, 0.005)
plt.show()


In [ ]:
np.min(y)

In [ ]:
np.max(y)

In [ ]:
np.std(y)

In [ ]:
# start = "2018-01-01"
# effr = web.DataReader("EFFR", "fred", start)   # Effective Federal Funds Rate
# effr.index = pd.to_datetime(effr.index)
# sofr = web.DataReader("SOFR", "fred", start)   # Secured Overnight Financing Rate
# sofr.index = pd.to_datetime(sofr.index)

# df = effr.join(sofr, how="inner").dropna()

# y = ((df["SOFR"] - df["EFFR"])/100.0).to_numpy()



In [ ]:
# Query data
# start = "2018-01-01"
# effr = web.DataReader("EFFR", "fred", start)   # Effective Federal Funds Rate
# sofr = web.DataReader("SOFR", "fred", start)   # Secured Overnight Financing Rate
# y = (sofr["SOFR"].dropna().to_numpy() / 100).copy()
# T = len(y)

# start = "2023-06-01"
# # series = "T10Y3M"   # term spread 10Y-2Y
# series = "BAMLH0A0HYM2"

# spread = web.DataReader(series, "fred", start)
# y_raw = spread[series].dropna().to_numpy() / 100.0  # đổi sang đơn vị 1
# y = y_raw #- y_raw.mean()                            # center cho dễ
# T = len(y)


# Test 2
# start = "2018-01-01"
# effr = web.DataReader("EFFR", "fred", start)   # Effective Federal Funds Rate
# effr.index = pd.to_datetime(effr.index)
# sofr = web.DataReader("SOFR", "fred", start)   # Secured Overnight Financing Rate
# sofr.index = pd.to_datetime(sofr.index)

# df = effr.join(sofr, how="inner").dropna()

# y = ((df["SOFR"] - df["EFFR"])/100.0).to_numpy()

# # Clean outlier
# idx_outlier = np.argmax(np.abs(y))
# y = np.delete(y, idx_outlier)
# T = len(y)

     
# # Measurements
# plt.figure(figsize=(8, 3.5))
# plt.title(f"Measurements ({error_type})")
# plt.plot(y, color="cornflowerblue")
# plt.xlabel(r"$t$")
# plt.ylabel(r"$y_t$")
# plt.tight_layout()

# pdf_add_fig()


In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

def is_stationary(x, alpha=0.05):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]

    adf_p = adfuller(x, autolag="AIC", regression="c")[1]     # H0: unit root (non-stationary)
    kpss_p = kpss(x, regression="c", nlags="auto")[1]         # H0: stationary

    return (adf_p < alpha) and (kpss_p > alpha), adf_p, kpss_p

In [ ]:
is_stationary(y)

In [ ]:
print(adfuller(y)[1])
print(adfuller(y)[1] < 0.05)


In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# y = your 1D array / pd.Series
z = y**2

# ACF
fig, ax = plt.subplots(figsize=(8, 4))
plot_acf(z, lags=50, ax=ax)
ax.set_title("ACF")
plt.tight_layout()
plt.show()

# (Optional) PACF
fig1, ax = plt.subplots(figsize=(8, 4))
plot_pacf(z, lags=50, ax=ax, method="ywm")
ax.set_title("PACF")
plt.tight_layout()
plt.show()

### 3.4 Check implementation (optional)
Check if the gradient and Hessian are correct by comparing to numerical differentiation.

In [ ]:
debug_gradient = False # False # NOTE: Make sure T is not too large if this is true. # WINSTON ADJUST
debug_Hessian = False # NOTE: Make sure T is not too large if this is true. # WINSTON ADJUST

if debug_gradient:
    assert T <= 10, "Set T smaller than 10 for debugging purposes."
    
    log_densities_for_numerical_gradient = lambda x: log_densities(hinv(x), y, model_name) # WINSTON ADJUST
        
    print("======================================================")
    print("Output to debug gradient (%s, %s):" % (model_name, error_type))
    print("======================================================")

    grad_l_t_all = grad_log_densities(theta0, y, model_name)  # WINSTON ADJUST
    print('Analytical gradient (first 2, last 2)')
    print(grad_l_t_all[:2, :])
    print(grad_l_t_all[-2:, :])

    
    # Debug the gradient by comparing to numerical differentiation.
    J_num = Jacobian(log_densities_for_numerical_gradient)(phi0)
    print('Numerical diff')
    print(J_num[:2, :])
    print(J_num[-2:, :])
    
    assert(np.allclose(grad_l_t_all, J_num))


if debug_Hessian:
    assert T <= 10, "Set T smaller than 10 for debugging purposes."

    Hess_num_func = lambda x: log_densities_ARMA_for_numerical_Hess(hinv(x), y[:t], model_name) # WINSTON ADJUST

    print("======================================================")
    print("Output to debug Hessian (%s, %s):" % (model_name, error_type))
    print("======================================================")
    
    # Debug Hessian
    Hess_l_t_all = Hess_log_densities(theta0, y, model_name) # WINSTON ADJUST
    print(Hess_l_t_all.shape)
    T = len(y)
    for t in range(1, T + 1):
        print('------------------------------------------------------')
        print('t = ', t)
        print('Analytical \n', Hess_l_t_all[t-1, :, :])
        # Hess_num_func = lambda x: log_densities_ARMA11(hinv(x), y)[t-1]   # WINSTON ADJUST
        Hess_num = Hessian(Hess_num_func)(phi0)
        print('Numerical diff \n', Hess_num)
        check_Hessian_close(Hess_l_t_all[t-1, :, :], Hess_num)

### 3.6 Time evaluations 
This subsection times the densities, gradients, and Hessians runs.

In [ ]:
with capture_output() as cap:
    print("==============================================================================================================")
    start = time.perf_counter()
    result = log_densities(theta0, y, model_name) # WINSTON ADJUST
    end = time.perf_counter()
    print("Elapsed time log-densities evaluation with T = " , T, "p = " , p, "q = " , q, ":", end - start, "seconds")

    start = time.perf_counter()
    result = grad_log_densities(theta0, y, model_name) # WINSTON ADJUST
    end = time.perf_counter()
    print("Elapsed time grad log-densities evaluation with T = " , T, "p = " , p, "q = " , q,  ":", end - start, "seconds")

    start = time.perf_counter()
    result = Hess_log_densities(theta0, y, model_name) # WINSTON ADJUST
    end = time.perf_counter()
    print("Elapsed time Hess log-densities evaluation with T = " , T, "p = " , p, "q = " , q,  ":", end - start, "seconds")
    print("==============================================================================================================")

text = cap.stdout
pdf_add_text(text, title="Time evaluations")
cap.show()

### 3.7 Maximum likelihood optimisation using the full data
We now perform optimisation to find the maximum likelihood estimate (MLE) to check if we can recover true parameters. The optimisation is performed on the unrestricted parameter $\boldsymbol{\phi}$.

The following code maximises the log-likelihood and constructs asymptotic confidence intervals (for $\boldsymbol{\phi}$). Note that although the optimisation function (obj_func_likelihood) for the log-likelihood is unrestricted, it does not inforce stationarity.

In [ ]:
# Minimise in phi space, so input is unrestricted
def log_penalty_phi(x):
    return log_penalty(hinv(x)) # WINSTON ADJUST

def obj_func_likelihood(x):
    """
    Penalised log-likelihood. To enforce the
    """
    return -log_likelihood(hinv(x), y, model_name)/T + log_penalty_phi(x)/T # WINSTON ADJUST

grad_log_penalty = grad(log_penalty_phi) #(x, p, q)
Hess_log_penalty = hessian(log_penalty_phi)

# WINSTON ADJUST
grad_obj_func_likelihood = lambda x: -grad_log_likelihood(hinv(x), y, model_name)/T + grad_log_penalty(x)/T # Gradient of obj_func_likelihood. Computed by automatic differentiation
Hess_obj_func_likelihood = lambda x: -Hess_log_likelihood(hinv(x), y, model_name)/T + Hess_log_penalty(x)/T # Hessian of obj_func_likelihood. Computed by automatic differentiation

optim_dir = out_dir / "optim"
optim_dir.mkdir(parents=True, exist_ok=True)
save_path_optim = optim_dir / "optim_likelihood_result.h5"

# Optimise the log-likelihood
start = time.perf_counter()
it = {"k": 0} # For callback
fg = FGCache(obj_func_likelihood, grad_obj_func_likelihood)

try:
    optim_results = load_dict_h5(save_path_optim)
    MLE = np.array(optim_results['optim_obj']['x'])
    Cov = optim_results['Cov_MLE']
    print('95% Confidence intervals for MLE estimates (phi)')
    print(np.round(list(MLE - 1.96*np.sqrt(np.diag(Cov))), 2))
    print(np.round(list(MLE + 1.96*np.sqrt(np.diag(Cov))), 2))
    print("\n\n")

except FileNotFoundError:
    theta_optim_start = np_autograd.array([0.7, 0.002, 0.003])

    phi_optim_start = h(theta_optim_start)
    
    start = time.perf_counter()
    it = {"k": 0} # For callback
    fg = FGCache(obj_func_likelihood, grad_obj_func_likelihood)

    start = time.perf_counter()
    it = {"k": 0} # For callback
    fg = FGCache(obj_func_likelihood, grad_obj_func_likelihood)

    if likelihood_optim == "trust-ncg":
        print('\n\nLikelihood optimisation settings for method trust-ncg:\n')
        options = options_trust_ncg
        print_kv(options)
        res_optim_likelihood = minimize(fg.fun, phi_optim_start, jac=fg.jac, hess=Hess_obj_func_likelihood, 
                                    method="trust-exact", options=options)  

    elif likelihood_optim == "L-BFGS-B":
        print('\n\nLikelihood optimisation settings for method L-BFGS-B:\n')
        options = options_L_BFGS_B
        print_kv(options)
        res_optim_likelihood = minimize(fg.fun, phi_optim_start, method='L-BFGS-B', jac= fg.jac, callback = callback_optim,
                                        options=options)
    elif likelihood_optim == "basinhopping":
        minimizer_kwargs = {"method": "L-BFGS-B", "jac": fg.jac, "options": options_L_BFGS_B}
        res_optim_likelihood = basinhopping(fg.fun, x0 = phi_optim_start, minimizer_kwargs=minimizer_kwargs,
                                            callback = callback_optim, **options_basinhopping)
    else:
        raise ValueError("Optimisation method does not exist")        

    end = time.perf_counter()
    run_time = end - start

    with capture_output() as cap:
        print("==============================================================")
        print("\nLikelihood optimisation took", end - start, "seconds.")        

        print('Starting values: \n', np.round(phi_optim_start, 3))
        # Compare answers
        print('True parameter values (phi)')
        print(np.round(phi0, 2))
        print('MLE estimates (phi)')
        MLE = res_optim_likelihood.x
        print(np.round(MLE, 2))
        print('True parameter values (theta)')
        print(theta0)
        print('hinv(MLE) estimates (theta)')
        print(np.round(hinv(MLE), 2))

        print('95% Confidence intervals for MLE estimates (phi)')
        Cov = np.linalg.inv(Hess_obj_func_likelihood(MLE))/T # Negative Hessian inverse evaluated at the mode
        print(np.round(list(MLE - 1.96*np.sqrt(np.diag(Cov))), 2))
        print(np.round(list(MLE + 1.96*np.sqrt(np.diag(Cov))), 2))
        print("\n\n")
        
        if likelihood_optim == "basinhopping":
            print_minimize_summary(res_optim_likelihood["lowest_optimization_result"], x_names=trans_param_names, latex=True)
        else:
            print_minimize_summary(res_optim_likelihood, x_names=trans_param_names, latex=True)

        print("==============================================================")
    text = cap.stdout
    pdf_add_text(text, title="Optimisation of log-likelihood. Summary.")
    cap.show()
    
    optim_results = {'optim_obj': pack_optimize_result(res_optim_likelihood), 'method': likelihood_optim, 'Cov_MLE': Cov, 'start_value': phi_optim_start}
    save_dict_h5(save_path_optim, optim_results)


### 3.8 Maximum aposteriori optimisation using the full data
We now perform optimisation to find the maximum aposterior (MAP) estimate. This is needed to construct our control variates, and is also useful for the random-walk MH proposal later. The optimisation is performed on the unrestricted parameter $\boldsymbol{\phi}$. Credible intervals for the (phi) parameters are provided, based on a normal approximation of the (transformed) posterior.

This step uses the MLE as starting value, so it is significantly faster than the optimisation that obtains the MLE (starting much closer to the mode). For this reason, we do not save this step to a hd5 file.

**NOTE**: 
The subsampling MCMC is only stable for cases when the negative Hessian at the posterior mode is positive definite. This need not hold for GARCH($p$,$q$) models, particularly when $q > 2$, due to weak identification among the volatility parameters. In such cases, we estimate the Fisher information by the sum of outer product of gradient observations. While this often results in a matrix that is positive definite, it may overestimate the posterior variation. This may cause issues since the proposal may be far from where the posterior mass is located, resulting in a potentially inaccurate control variate. We have employed samplers that, when the Fisher is estimated this way, start with a small scaling factor and adapt it to have a target acceptance probability of 0.23. While this may help subsampling MCMC, it is not guaranteed to work. Take away message: subsampling MCMC works reliaibly for cases where the observed Fisher information is invertible. 

In [ ]:
# Define gradients and Hessian for log-posterior
log_prior_phi = lambda x: log_prior(hinv(x)) + logdet(x) # for autograd  # WINSTON ADJUST
grad_log_prior_phi = grad(log_prior_phi)
grad_log_posterior = lambda x: grad_log_likelihood(hinv(x), y, model_name) + grad_log_prior_phi(x) # WINSTON ADJUST
Hess_log_prior_phi = hessian(log_prior_phi)
Hess_log_posterior = lambda x: Hess_log_likelihood(hinv(x), y, model_name) + Hess_log_prior_phi(x) # WINSTON ADJUST

obj_func_posterior = lambda x: -log_posterior(x, y, model_name)[0] # Input is phi  # WINSTON ADJUST
grad_obj_func_posterior = lambda x: -grad_log_posterior(x) 
Hess_obj_func_posterior = lambda x: -Hess_log_posterior(x)

# Optimise the log-posterior in phi space, so input is unrestricted
phi_optim_start = MLE
start = time.perf_counter()
it = {"k": 0} # For callback
fg = FGCache(obj_func_posterior, grad_obj_func_posterior)

res_optim_posterior = minimize(fg.fun, phi_optim_start, method='L-BFGS-B', jac= fg.jac, callback = callback_optim,
                                options=options_L_BFGS_B)
end = time.perf_counter()
with capture_output() as cap:
    print("==============================================================")

    print("\nPosterior optimisation took", end - start, "seconds.")
    print('\n\nPosterior optimisation settings for method L-BFGS-B:\n')
    print_kv(options_L_BFGS_B)
    
    print('Starting values: \n', np.round(phi_optim_start, 3))
    # Compare answers
    print('True parameter values (phi)')
    print(np.round(phi0, 2))
    print('MAP estimates (phi)')
    MAP = res_optim_posterior.x
    print(np.round(MAP, 2))
    print('True parameter values (theta)')
    print(theta0)
    print('hinv(MAP) estimates (theta)')
    print(np.round(hinv(MAP), 2))

    print('95% Credible intervals (normality assumption on posterior) (phi)')
    Hess_MAP = Hess_obj_func_posterior(MAP)
    Cov = np.linalg.inv(Hess_MAP) # Negative Hessian inverse evaluated at the mode  
    if (np.linalg.eigvalsh(Cov) < 0).any():
        print("Cannot compute posterior covariance from Hessian. Trying outer product approximation")
        scores = grad_log_densities(hinv(MAP), y, model_name) # WINSTON ADJUST
        Cov = np.linalg.inv(scores.T @ scores)
        assert (np.linalg.eigvalsh(Cov) > 0).all(), "Cannot compute posterior covariance."
        adapt_scaling = True
    else:
        adapt_scaling = False
    print(np.round(list(MAP - 1.96*np.sqrt(np.diag(Cov))), 2))
    print(np.round(list(MAP + 1.96*np.sqrt(np.diag(Cov))), 2))
    print("\n\n")
    print_minimize_summary(res_optim_posterior, x_names=trans_param_names, latex=True)
    
    print("==============================================================")

    
text = cap.stdout
pdf_add_text(text, title="Optimisation of log-posterior. Display and summary.")
cap.show()


### 3.9 Sampling the full data posterior (NOTE: Maybe computationally expensive for large $T$)
This section uses a Metropolis-Hastings sampler to sample from the the full data posterior distribution. This will serve as the ground truth to compare our subsampling MCMC and variational Bayes intractable log-likelihood results against. It will also help us establish the support of the posterior to evaluate the quality of the control variates.

In [ ]:
# Set seed
np.random.seed(1999)

# MCMC settings. Samples and prop_cov
N = 5500 # MCMC samples
MCMC_dir = out_dir / "MCMC"
MCMC_dir.mkdir(parents=True, exist_ok=True)
save_path_MCMC = MCMC_dir / ("full_data_MCMC_N%s.h5" % N)

try:
    MCMC_results = load_dict_h5(save_path_MCMC)
    result_full_data_MCMC = MCMC_results['MCMC_obj']
    prop_cov = MCMC_results['prop_cov']
    # Convert back to arrays
    for key in ["samples_phi", "samples_theta"]:
        result_full_data_MCMC[key] = np.asarray(result_full_data_MCMC[key])

except FileNotFoundError:
    # Run MCMC
    Sigma_pi = Cov # Negative Hessian inverse evaluated at the mode
    prop_cov = 2.38**2/n_params*Sigma_pi # For the random walk Negative Hessian inverse evaluated at the mode

    phi_init = MAP # + sps.norm.rvs(0, 0.1*np.sqrt(np.diag(Sigma_pi)), size = len(MAP))
    names = {"params_name": param_names, "trans_param_names": trans_param_names}
    print("==============================================================")
    result_full_data_MCMC = sample_full_data_posterior_random_walk(phi_init, prop_cov, N, names, dataset_name, adapt = adapt_scaling)
    print("==============================================================")

    MCMC_results = {'MCMC_obj': result_full_data_MCMC, 'prop_cov': prop_cov, 'start_value': phi_init}
    save_dict_h5(save_path_MCMC, MCMC_results)


In [ ]:
# Plots
plot_credible_intervals(result_full_data_MCMC['samples_phi'], param_true=None, param_names=trans_param_names, use_median=False)
plot_credible_intervals(result_full_data_MCMC['samples_theta'], param_true=None, param_names=param_names, use_median=False)
plot_traceplots_MCMC(result_full_data_MCMC['samples_phi'], 
                param_true = None, param_names=trans_param_names,
                burnin=0, thin=1, use_median=False, show_running_mean=True, runmean_window=50, 
                suptitle="Traceplots full data MCMC")
plot_traceplots_MCMC(result_full_data_MCMC['samples_theta'], 
                param_true = None, param_names=param_names,
                burnin=0, thin=1, use_median=False, show_running_mean=True, runmean_window=50, 
                suptitle="Traceplots full data MCMC")
pdf_add_fig()

### 3.10 Initialise and evaluate control variates

In [ ]:
phiStar = MAP
args = (y, model_name)  # WINSTON ADJUST
control_variate_dir = out_dir / "control_variates"
control_variate_dir.mkdir(parents=True, exist_ok=True)
save_path_control_variates = control_variate_dir / "control_variate_result.h5"
try:
    control_variates = load_dict_h5(save_path_control_variates)
    dens_at_phiStar = control_variates['dens_at_phiStar']
    grad_at_phiStar = control_variates['grad_at_phiStar'] 
    Hess_at_phiStar = control_variates['Hess_at_phiStar']
    end = control_variates['end']
    start = control_variates['start']

except FileNotFoundError:
    start = time.perf_counter()
    dens_at_phiStar,  grad_at_phiStar, Hess_at_phiStar = initiate_control_variate_quantities(log_densities, grad_log_densities, Hess_log_densities, phiStar, args)
    end = time.perf_counter()
    text = cap.stdout
    control_variates = {'dens_at_phiStar': dens_at_phiStar, 'grad_at_phiStar': grad_at_phiStar, 
                        'Hess_at_phiStar': Hess_at_phiStar, 'end': end, 'start': start}
    save_dict_h5(save_path_control_variates, control_variates)


with capture_output() as cap:
    print("==============================================================")
    print("Elapsed time control variates initialisation with T = " , T, "p = " , p, "q = " , q,  ":", end - start, "seconds")
    print("==============================================================")
text = cap.stdout
pdf_add_text(text, title="Control variate construction.")
cap.show()

# Precompute quantities for the control variates that do not depend on theta
A = np.sum(dens_at_phiStar) 
B = np.sum(grad_at_phiStar, axis = 0)
C = np.sum(Hess_at_phiStar, axis = 0)
# sum of control variates evaluated at phi    
q_sum = lambda phi: A + np.dot(B, phi - phiStar) + 0.5*np.dot(phi - phiStar, np.dot(C, phi - phiStar))


The accuracy of the control variates is evaluated for different orders (0, 1, 2) of the Taylor approximation. The control variates are evaluated using samples $\boldsymbol{\phi}$ from the normal approximation of the posterior distribution. The samples are constructed to be at increasingly further distances from the posterior mode. 

In [ ]:
# Outside median ellipse in phi space
Sigma_pi = Cov # np.linalg.inv(Hess_obj_func_posterior(MAP))
phi = mvn_tail_sample(MAP, Sigma_pi, q = 0.50, n = 1).flatten()
phi_beyond_median = phi
q_k_order0 = eval_q_k(phi, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 0) 
q_k_order1 = eval_q_k(phi, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 1) 
q_k_order2 = eval_q_k(phi, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 2) 
l_k = log_densities(hinv(phi), *args)

fig, axs = plt.subplots(1, 3, constrained_layout=True, figsize=(8.5, 3.2))
fig.suptitle(
    r'%s(%s, %s) %s. $\phi$ outside median ellipse of $\phi$' % (model_name, p, q, error_type) + r'(, $||\phi - \phi^\star|| = %3.4f)$' % np.linalg.norm(phi - phiStar),
    fontsize=14, y=0.98   # try 0.99 if you want it even closer
)
fig.set_constrained_layout_pads(h_pad=0.4, w_pad=0.05, hspace=0.08, wspace=0.08)
for k in range(3):
        axs[k].plot(l_k, l_k, color = 'red')
        if k == 0:
            axs[k].plot(l_k, q_k_order0, '.', color = 'cornflowerblue')
            axs[k].set_title('Taylor order 0', size = 12)
        elif k == 1:
            axs[k].plot(l_k, q_k_order1, '.', color = 'cornflowerblue')
            axs[k].set_title('Taylor order 1', size = 12)
        elif k == 2:
            axs[k].plot(l_k, q_k_order2, '.', color = 'cornflowerblue')
            axs[k].set_title('Taylor order 2', size = 12)
            
        axs[k].axis('scaled')


# Outside 0.999 ellipse
phi = mvn_tail_sample(MAP, Sigma_pi, q = 0.999, n = 1).flatten()
phi_extreme = phi
q_k_order0 = eval_q_k(phi, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 0) 
q_k_order1 = eval_q_k(phi, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 1) 
q_k_order2 = eval_q_k(phi, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 2) 
l_k = log_densities(hinv(phi), *args)

fig, axs = plt.subplots(1, 3, constrained_layout=True, figsize=(8.5, 3.2))
fig.suptitle(
    r'%s(%s, %s) %s. $\phi$ outside 0.999 ellipse of $\phi$' % (model_name, p, q, error_type) + r'(, $||\phi - \phi^\star|| = %3.4f)$' % np.linalg.norm(phi - phiStar),
    fontsize=14, y=0.98   # try 0.99 if you want it even closer
)
fig.set_constrained_layout_pads(h_pad=0.4, w_pad=0.05, hspace=0.08, wspace=0.08)
for k in range(3):
        axs[k].plot(l_k, l_k, color = 'red')
        if k == 0:
            axs[k].plot(l_k, q_k_order0, '.', color = 'cornflowerblue')
            axs[k].set_title('Taylor order 0', size = 12)
        elif k == 1:
            axs[k].plot(l_k, q_k_order1, '.', color = 'cornflowerblue')
            axs[k].set_title('Taylor order 1', size = 12)
        elif k == 2:
            axs[k].plot(l_k, q_k_order2, '.', color = 'cornflowerblue')
            axs[k].set_title('Taylor order 2', size = 12)
            
        axs[k].axis('scaled')


### 3.11 Truncated power-law decaying sampling probabilities
Compute and plot the probability mass function for two cases. $\gamma = 0$ (uniform weights) and $\gamma > 0$.

In [ ]:
# Check probability mass function
# Case 1: gamma = 0. Should provide uniform weights
t_star = int(0.03*T) # Hyperparameter
b = 100.0  # Hyperparameter

# Compute pmf
gamma_val = 0.0
probs, eps = TPD_probs(T,t_star, gamma = gamma_val, b = b)

# Plot the pmf
plt.figure(figsize=(9,4))
plt.plot(range(1, T+1), probs, marker='o', markersize=2)
plt.axvline(t_star+0.5, color='red', linestyle='--', label='t*')
plt.xlabel("t")
plt.ylabel(r"$p_t$")
plt.title(r"Case 1: Probability mass function TPD when $\gamma = %3.3f$" % gamma_val)
plt.legend()

# Print diagnostics
print("==============================================================================================================")
print("Case 1:")
print("T = ,", T, "tstar = ", t_star)
print("Case gamma = ", gamma_val)
print("Head mass (empirical):", probs[:t_star].sum())
print("Tail mass (empirical):", probs[t_star:].sum())
print("Head min prob:", probs[:t_star].min())
print("Tail min prob:", probs[t_star:].min())
print("Reference: Uniform min:", 1/T)
print("Tail ratio (baseline uniform)", probs[t_star:][0]/(1/T))
print("Reference: eps from function:", eps)
print("Which c: ", T/(T - t_star)*eps)
print("Sum of probs:", probs.sum())

# Case 2
# Compute pmf
gamma_val = 14.0
probs, eps = TPD_probs(T, t_star, gamma = gamma_val, b = b)

# Plot the pmf
plt.figure(figsize=(9,4))
plt.plot(range(1, T+1), probs, marker='o', markersize=2)
plt.axvline(t_star+0.5, color='red', linestyle='--', label='t*')
plt.xlabel("t")
plt.ylabel(r"$p_t$")
plt.title("Case 2: Probability mass function TPD when $\gamma = %3.3f$" % gamma_val)
plt.legend()

# Print diagnostics
print("\n")
print("Case 2:")
print("Case gamma = ", gamma_val)
print("Head mass (empirical):", probs[:t_star].sum())
print("Tail mass (empirical):", probs[t_star:].sum())
print("Head min prob:", probs[:t_star].min())
print("Tail prob:", probs[t_star:][0])
print("Reference: Uniform min:", 1/T)
print("Tail ratio (baseline uniform)", probs[t_star:][0]/(1/T))
print("Reference: eps from function:", eps)
print("Which c: ", T/(T - t_star)*eps)
print("Sum of probs:", probs.sum())
print("==============================================================================================================")

### 3.12 Compute variance and expected computational cost
This section shows how the variance and expected computational cost are computed

In [ ]:
# # MAP and Perturbed MAP
phi_MAP = MAP
phi_MAP_perturbed = MAP + sps.norm.rvs(scale = 0.01*np.diag(Sigma_pi), size = n_params)


### 3.13 Subsampling MCMC
This section shows how to run subsampling MCMC. We start by tuning the algorithm, i.e. choosing with $m$ and $c$ to use for constructing the sampling weights. Note that setting a $c$ implies the largest $\gamma_{\mathrm{max}}$ compatible with the floor $\varepsilon(\gamma) \geq c(T-t^\star)/T$; see the paper for details. 

We consider two distinct approaches for the tuning:

1. Minimising the variance of the estimator under a computational constraint.
2. Minimising the computational cost of the estimator under a variance constraint.

Both approaches require the population $e_t =  \ell_t - q_t$, which requires knowledge of $\boldsymbol{\theta}$. We tune the algorithm based on a typical $\boldsymbol{\theta}$, such as one close to the posterior maximum aposteriori (MAP) estimate. For the first approach, we consider an additional tuning strategy that does not depend on the $e_t$, because it minimises an upper bound of the variance. See the paper for details.

The natural approach for subsampling MCMC is to minimise compute given a variance tolerance budget (approach 2), as the literature provides guidelines on what an optimal variance (around 1) should be.

In [ ]:
# for V in [2e-10, 1e-10]:
#     c_tmp, m_tmp = tune_c_m_approach2(V, e_t_tune_approach2)
#     print("V_tolerance =", V, "=> m_opt =", m_tmp, ", c_opt =", c_tmp)

In [ ]:
# First tune the algorithm. Tune around theta_MAP_perturbed
phi_tune_approach2 = phi_MAP_perturbed
q_k_order2 = eval_q_k(phi_tune_approach2, phiStar, dens_at_phiStar, grad_at_phiStar, Hess_at_phiStar, order = 2)
e_t_tune_approach2 = log_densities(hinv(phi_tune_approach2), *args) - q_k_order2 # Population elements to tune at
V_tolerance = 1 # WINSTON ADJUST
c_opt, m_opt = tune_c_m_approach2(V_tolerance, e_t_tune_approach2)
# b = 100
# m_opt = 6

gamma_opt = gamma_max_value(T, t_star, c_opt, b)
probs, eps = TPD_probs(T, t_star, gamma = gamma_opt, b = b)
with capture_output() as cap:
    print("==============================================================")
    print("Tuning approach 2 (minimise expected compute given variance tolerance)")
    print("c_opt =", c_opt, "m_opt =", m_opt)
    print("Implied gamma_max =", np.round(gamma_opt))    
    print("==============================================================")
    
text = cap.stdout
pdf_add_text(text, title="Tuning of subsampling MCMC.")
cap.show()

# Plot optimal probabilities
# Compute pmf
probs, eps = TPD_probs(T, t_star, gamma = gamma_opt, b = b)
# Plot the pmf
plt.figure(figsize=(9,4))
plt.plot(range(1, T+1), probs, marker='o', markersize=2)
plt.axvline(t_star+0.5, color='red', linestyle='--', label='t*')
plt.xlabel("t")
plt.ylabel(r"$p_t$")
plt.title(r"Approach 2 tune when $V = %3.3f$: Pmf TPD when $\gamma = %3.3f$" % (V_tolerance, gamma_opt))
plt.legend()
pdf_add_fig()



In [ ]:
# Set seed
np.random.seed(2026)

N = 32000
save_path_subsampling_MCMC = MCMC_dir / ("subsampling_MCMC_N%s.h5" % N)

try:
    subsampling_MCMC_results = load_dict_h5(save_path_subsampling_MCMC)
    result_subsampling_MCMC = subsampling_MCMC_results['MCMC_obj']
    prop_cov = subsampling_MCMC_results['prop_cov']
    # Convert back to arrays
    for key in ["samples_phi", "samples_theta"]:
        result_subsampling_MCMC[key] = np.asarray(result_subsampling_MCMC[key])

except FileNotFoundError:
    # Run Subsampling MCMC
    estimator_params = {"m_opt": m_opt, "c_opt": c_opt, "gamma_opt": gamma_opt, "t_star": t_star, "b":b, 
                        "tuning_approach": "Approach 2", "tuning_budget": V_tolerance}
    phi_init = MAP 
    prop_cov = 2.38**2/n_params*Sigma_pi # For the random walk Negative Hessian inverse evaluated at the mode
    names = {"params_name": param_names, "trans_param_names": trans_param_names}
    print("==============================================================")
    result_subsampling_MCMC = sample_subsampling_posterior_random_walk(estimator_params, phi_init, prop_cov, N, names, dataset_name, adapt = adapt_scaling)
    print("==============================================================")
    subsampling_MCMC_results = {'MCMC_obj': result_subsampling_MCMC, 'estimator_params': estimator_params, 'prop_cov': prop_cov, 'start_value': phi_init}
    save_dict_h5(save_path_subsampling_MCMC, subsampling_MCMC_results)




In [ ]:
# Plot results
burn_in = 500
plot_credible_intervals(result_subsampling_MCMC['samples_phi'], param_true=None, param_names=trans_param_names, use_median=False)
plot_credible_intervals(result_subsampling_MCMC['samples_theta'], param_true=None, param_names=param_names, use_median=False)
plot_traceplots_MCMC(result_subsampling_MCMC['samples_phi'], 
                param_true = None, param_names=trans_param_names,
                burnin=0, thin=1, use_median=False, show_running_mean=True, runmean_window=50, 
                suptitle = "Traceplots subsampling MCMC")
plot_subsampling_MCMC_quantities(result_subsampling_MCMC)

tau_full = np.array([IACT(result_full_data_MCMC['samples_phi'][burn_in:, item]) for item in range(n_params)])
tau_sub = np.array([IACT(result_subsampling_MCMC['samples_phi'][burn_in:, item]) for item in range(n_params)])
plot_inefficiency_comparison(tau_full, tau_sub, param_names = trans_param_names)

tau_full = np.array([IACT(result_full_data_MCMC['samples_theta'][burn_in:, item]) for item in range(n_params)])
tau_sub = np.array([IACT(result_subsampling_MCMC['samples_theta'][burn_in:, item]) for item in range(n_params)])
plot_inefficiency_comparison(tau_full, tau_sub, param_names = param_names)


plot_kde_comparison(result_full_data_MCMC['samples_phi'][burn_in:, :], result_subsampling_MCMC['samples_phi'][burn_in:, :], 
                    param_names = trans_param_names, title="Posterior marginals in phi-space: Full vs Subsampling MCMC", 
                    n_points=200, true_vals=None, labels=("Full MCMC", "Subsampling"))
plot_kde_comparison(result_full_data_MCMC['samples_theta'][burn_in:, :], result_subsampling_MCMC['samples_theta'][burn_in:, :], 
                    param_names = param_names, title="Posterior marginals in theta-space: Full vs Subsampling MCMC", 
                    n_points=200, true_vals=None, labels=("Full MCMC", "Subsampling"))

pdf_add_fig()
_pdf_close() # Close the pdf that contains the generated figures and text.  # WINSTON ADJUST
